# LB 0.477 · Two 0.472 Fixes, Merged the Right Way

Two public changes to the same 0.468 leader-safe recipe each reach a displayed 0.472: Roman Tamrazov's fragment check, and alex chilton's heavier neural weight on ranks 2-25 (shown by its fork; his own page shows 0.467). Stacked the plain way they still score 0.472; our explanation is that the fragment check's leader rule picks its contenders from the order it is handed. Run on the order it was built on, with ranks 2-25 from the 0.75 order, the two add up.

| | |
|---|---|
| result | **public LB 0.477 (submission 57722897, read 2026-10-29 06:05 UTC)** |
| beats | every public notebook on the Code tab at 2026-10-28 07:48 UTC: [Heliosli](https://www.kaggle.com/code/heliosli/happy-to-make-the-dataset-public-if-it-helps) at 0.475 (0.002 below ours, about half of one rank-2-to-1 move; it needs a private dataset), then ten notebooks at 0.472, among them [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax) and the w_mid 0.70-0.80 forks |
| runtime | commit run 21-30 min in our runs (12 molecules; m1's took 21); a scoring run takes hours on Kaggle T4, internet off |


**The task in one paragraph.** A tandem mass spectrum (MS/MS) records the pieces a molecule breaks into in a collision
cell: a list of fragment masses with their intensities. For each hidden molecule a notebook hands in up to 25
structures (SMILES), best guess first. The score is MRR@25: 1 for a right first guess, 1/2 for a right second guess,
down to 1/25, and 0 when the right structure is not in the list, averaged over molecules. An answer counts as right
when its atom connectivity matches (the first 14 characters of the InChIKey after RDKit's tautomer canonicalisation),
so a wrong stereo centre or tautomer is not penalised.

**What this notebook is.** Roman Tamrazov's [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax) (version 22) with his provenance reranker switched off and one
change in how its fragment check is used. The check's rank-1 rule runs on the 0.468 order it was built on (V2,
ranks 2-25 at neural weight 0.50), and only its leader moves are kept; ranks 2-25 come from alex chilton's order at
0.75. This is run m1, the best public score of our 13 (section 5), tied with m2, m4, m6. The code that builds the answer lists is his and the code he pins, plus one merge cell of
ours; four of his cells carry our string patches, listed in section 6. What is ours: the explanation that the two
public 0.472 changes collide in the fragment check's contender window (m1 tests it together with dropping the check's
swaps and last-place change, not alone), the merge that keeps them apart, the
13 hidden-test submissions that measured it, the field record and the per-layer audit of this run.

**Reading map.** 1 The field · 2 One lineage, two branches · 3 The pipeline · 4 What this run did · 5 Every
submission of this pipeline · 6 Run it yourself · 7 What we concluded · 8 Credits

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
VARIANT = 'CUSTOM'
#   V0  baseline 0.465 answers, no neural term
#   V1  public 0.466 recipe: neural reciprocal-rank fusion, weight 0.25 everywhere
#   V2  leader-safe consensus: rank 1 as V1, ranks 2-25 at neural weight 0.50   (upstream default)
#   V3  V2 + tail rescue: outside candidates may take the last 3 places by neural score
#   V4  bold consensus: rank 1 as V1, ranks 2-25 at neural weight 1.00
#   'CUSTOM' uses the weights below (w_top=0 keeps the pipeline's own rank 1)
CUSTOM = dict(w_top=0.25, w_mid=0.75, rescue=0)

RRF_OFFSET = 3.0              # k in 1/(k + rank), as in the public recipe
COMMIT_MODE = 'smoke'         # 'smoke': the commit run ranks 12 visible molecules and draws every figure
                              # 'fast' : the commit run skips the pipeline (saves ~30 GPU minutes);
                              #          the scoring rerun on the hidden test always runs in full
OUTSIDERS_PER_MOLECULE = 20   # candidate pool for the tail rescue (V3, or CUSTOM with rescue > 0)
TIME_GUARD_HOURS = 7.5        # optional stages are skipped once the session has run this long
QUIET_LOGS = True             # one line per pipeline stage here; full logs stay in /kaggle/working/*.log
ENABLE_TAIL_SLOT = True
PHYSICAL_POST_SECONDS = 1500

# This notebook: CUSTOM, rank 1 at neural weight 0.25, ranks 2-25 at 0.75. PHYSICAL switches the fragment check,
# PROVENANCE the provenance reranker; neural model: Saner Turhaner v1.
PHYSICAL = True
PROVENANCE = False
# variant M: the fragment check reads variants/submission_V2.csv (cell 5 as upstream, the input of Spectral Parallax V17);
# the merge cell after the provenance cell keeps only its rank-1 moves, on the CUSTOM order
FRAG_ON_V2 = True   # False leaves submission.csv as the fragment check wrote it (V2 order)


In [ ]:
# ── Session: paths, run mode, design tokens and small display helpers ─────────────────────────
import os, sys, re, io, json, time, math, base64, shutil, hashlib, subprocess, collections, warnings
from pathlib import Path

NB_T0 = time.time()
warnings.filterwarnings('ignore')
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
CODE = ROOT / 'p2s_code'; CODE.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import display, HTML, FileLink

# Design tokens: chart ink and a fixed categorical order (identity never follows rank).
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, SURFACE, CARD = '#e1e0d9', '#c3c2b7', '#fcfcfb', '#f7f8fb'
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948')
QUIET_GRAY = '#c9c8c1'
ROLE = {'main': BLUE, 'engine2': ORANGE, 'pubchem': AQUA, 'new': VIOLET}
FONT = "system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10.5, 'axes.titlesize': 12.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.titlecolor': INK, 'axes.titlepad': 10, 'axes.labelcolor': INK2, 'axes.edgecolor': AXIS,
    'axes.linewidth': 0.8, 'xtick.color': AXIS, 'ytick.color': AXIS, 'xtick.labelcolor': INK2,
    'ytick.labelcolor': INK2, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'grid.linestyle': '-', 'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'legend.fontsize': 10, 'figure.dpi': 110, 'savefig.dpi': 160,
    'savefig.bbox': 'tight'})
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass


def tiles(items):
    """A row of stat tiles: items = [(label, value, note), ...]."""
    cells = ''.join(
        f'<div style="display:inline-block;vertical-align:top;min-width:150px;margin:0 10px 10px 0;'
        f'padding:12px 16px;border-radius:12px;background:#ffffff;border:1px solid rgba(11,11,11,0.10);">'
        f'<div style="font-size:12px;color:{INK2};">{label}</div>'
        f'<div style="font-size:25px;font-weight:650;color:{INK};line-height:1.3;">{value}</div>'
        f'<div style="font-size:11.5px;color:{MUTED};">{note}</div></div>' for label, value, note in items)
    display(HTML(f'<div style="font-family:{FONT};margin:4px 0 2px;">{cells}</div>'))


def note(title, text, color=VIOLET):
    display(HTML(f'<div style="font-family:{FONT};border-left:4px solid {color};background:{CARD};'
                 f'padding:10px 14px;border-radius:8px;margin:8px 0;font-size:13.5px;line-height:1.5;">'
                 f'<b style="color:{INK};">{title}</b> <span style="color:{INK2};">{text}</span></div>'))


def table_html(frame, highlight=None, numeric=()):
    """Plain, readable HTML table; numbers right-aligned; `highlight` = index label of a row to emphasise."""
    align = {c: 'right' if c in set(numeric) else 'left' for c in frame.columns}
    head = ''.join(f'<th style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {AXIS};'
                   f'color:{INK2};font-weight:600;background:transparent;">{c}</th>' for c in frame.columns)
    body = []
    for idx, row in frame.iterrows():
        on = highlight is not None and idx == highlight
        style = 'background:#efedfb;font-weight:600;' if on else 'background:transparent;'
        body.append(f'<tr style="{style}">' + ''.join(
            f'<td style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {GRID};color:{INK};'
            f'font-variant-numeric:tabular-nums;">{v}</td>' for c, v in zip(frame.columns, row.values)) + '</tr>')
    display(HTML(f'<table style="font-family:{FONT};font-size:13px;border-collapse:collapse;margin:6px 0 10px;">'
                 f'<thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>'))


def minutes(seconds):
    return f'{seconds / 60:,.1f} min' if seconds >= 60 else f'{seconds:,.0f} s'


def gpu_name():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=30).stdout.strip()
        return ' + '.join(x.strip() for x in out.splitlines() if x.strip()) or 'none'
    except Exception:
        return 'none'


# Run mode: the commit run sees the visible format example; the scoring rerun sees the hidden test.
COMP = next((p for p in (INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-casmi26-molecule-id-mass-spectra')
             if all((p / f).is_file() for f in ('train.parquet', 'test.parquet', 'sample_submission.csv'))), None)
assert COMP is not None, 'Attach the competition data (enveda-CASMI26-molecule-id-mass-spectra)'
_ids = pd.read_parquet(COMP / 'test.parquet', columns=['spectrum_id']).spectrum_id.astype(str)
TEST_SIGNATURE = hashlib.md5(','.join(sorted(_ids)).encode()).hexdigest()
del _ids
IS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or TEST_SIGNATURE != '654e030e4173780e945252b4a5adf70b'
RUN_PIPELINE = IS_RERUN or COMMIT_MODE != 'fast'
SHOW = not IS_RERUN                      # figures are drawn for readers; the scoring rerun skips them
SAMPLE_IDS = pd.read_csv(COMP / 'sample_submission.csv', dtype=str).molecule_id.tolist()
assert VARIANT in ('V0', 'V1', 'V2', 'V3', 'V4', 'CUSTOM'), VARIANT

GPU = gpu_name()
MODE = 'Scoring rerun' if IS_RERUN else ('Commit · smoke run' if RUN_PIPELINE else 'Commit · fast path')
tiles([('Run mode', MODE, 'hidden test' if IS_RERUN else 'visible format example'),
       ('Variant', VARIANT, 'set in the configuration cell'),
       ('Accelerator', GPU.split(' + ')[0] if GPU != 'none' else 'none', GPU if ' + ' in GPU else 'GPU required'),
       ('Python', '.'.join(map(str, sys.version_info[:3])), 'pipeline expects 3.12'),
       ('Answer rows', f'{len(SAMPLE_IDS):,}', 'molecules in sample_submission')])

## 1. The field

On 2026-10-29 06:00 UTC the Code tab listed 1009 public notebooks for this competition, 697 of them
with a public score. Of those, 133 score 0.460 or more, and 122 of these 133 attach the
same dataset, Dmitrii Gluzdov's v4g reproduction assets (Figure 1). Most of the top of the Code tab is one pipeline
with different constants.

The leaderboard on 2026-10-29 06:04 UTC had 4,295 teams. Of these, 231 sat at exactly 0.477 (the
score of this notebook, which 11 public notebooks now display), 175 at 0.472, 107 at
0.468 and 158 at 0.464. The board does not record what each team submitted, so this matches scores and does not prove
copying. The gold line was rank 18 at 0.481, and first place was 0.516. No member of the top
ten teams is the author of any of the 1009 public notebooks.

In [ ]:
# House style for our own figures (engine/33_style.py at repo commit 30b76796, without its ink and grid
# colours, so the upstream tiles and tables keep theirs). A later cell imports another `apply`; ours stays as style_apply.
ACCENT = "#d9480f"      # the one colour that means "this is the point"
SECOND = "#1c7ed6"      # used only when two things must be told apart
PALETTE = [ACCENT, SECOND, "#495057", "#adb5bd", "#2b8a3e", "#862e9c"]


def apply() -> None:
    """Set matplotlib defaults. Safe to call more than once."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (9, 4.6), "figure.dpi": 110, "savefig.dpi": 160,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
        "axes.labelsize": 11, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "axes.prop_cycle": mpl.cycler(color=PALETTE),
        "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 10,
    })


def label_point(ax, x, y, text: str, color: str = ACCENT) -> None:
    """Print the number that matters on the chart, next to the point it describes."""
    ax.annotate(text, (x, y), xytext=(6, 6), textcoords="offset points", color=color,
                fontsize=11, fontweight="semibold")


def first_screen(claim: str, number: str, beats: str | None = None, runtime: str | None = None,
                 accel: str | None = None) -> str:
    """The markdown every busyaprime notebook opens with. Returns it so a builder can write it into cell 0.

    claim   one sentence: what this notebook shows that the reader did not know
    number  the result, with its metric and where it was measured (public LB, CV on stated folds)
    beats   the public work it beats, with that work's number and a link
    runtime wall clock on the stated accelerator
    """
    rows = [f"| result | **{number}** |"]
    if beats:
        rows.append(f"| beats | {beats} |")
    if runtime:
        rows.append(f"| runtime | {runtime}{' on ' + accel if accel else ''} |")
    return f"{claim}\n\n| | |\n|---|---|\n" + "\n".join(rows) + "\n"
style_apply = apply

In [ ]:
# The field as the Code tab showed it on 2026-10-28 07:48 UTC: every public notebook with a displayed score.
FIELD = {"taken_utc":"2026-10-28 07:48","notebooks_listed":974,"scored":[[0.475,44,"E",1,1,"2026-10-22"],[0.472,92,"M",1,1,"2026-10-23"],[0.472,66,"E",1,1,"2026-10-25"],[0.472,8,"C",1,1,"2026-10-27"],[0.472,1,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.471,0,"C",1,1,"2026-10-27"],[0.471,0,"C",1,1,"2026-10-28"],[0.471,0,"C",1,1,"2026-10-28"],[0.47,0,"C",1,1,"2026-10-26"],[0.47,4,"C",1,1,"2026-10-26"],[0.468,61,"E",1,0,"2026-09-28"],[0.468,98,"M",1,1,"2026-10-13"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"E",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-27"],[0.467,59,"E",1,1,"2026-10-25"],[0.467,1,"C",1,1,"2026-10-25"],[0.467,0,"E",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,1,"C",0,0,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,3,"C",1,1,"2026-10-26"],[0.466,2,"E",1,1,"2026-09-17"],[0.466,80,"E",1,1,"2026-10-21"],[0.466,21,"C",1,1,"2026-10-24"],[0.466,6,"E",1,0,"2026-10-24"],[0.466,1,"E",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"M",1,1,"2026-10-25"],[0.466,1,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,0,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-27"],[0.465,182,"M",1,0,"2026-10-16"],[0.465,22,"C",1,0,"2026-10-24"],[0.465,0,"C",1,0,"2026-10-24"],[0.465,3,"C",1,0,"2026-10-24"],[0.465,1,"C",1,0,"2026-10-24"],[0.465,0,"C",1,1,"2026-10-25"],[0.465,1,"C",1,1,"2026-10-25"],[0.465,1,"C",1,1,"2026-10-25"],[0.465,0,"M",1,1,"2026-10-26"],[0.465,0,"C",1,0,"2026-10-26"],[0.465,0,"C",1,1,"2026-10-26"],[0.465,0,"E",0,1,"2026-10-28"],[0.464,201,"M",1,0,"2026-09-15"],[0.464,3,"C",1,0,"2026-10-22"],[0.464,2,"C",1,0,"2026-10-22"],[0.464,1,"C",1,0,"2026-10-22"],[0.464,1,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,2,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,3,"C",1,0,"2026-10-23"],[0.464,5,"C",1,0,"2026-10-23"],[0.464,10,"M",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,2,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-24"],[0.464,1,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,1,"C",1,1,"2026-10-26"],[0.464,1,"E",0,1,"2026-10-28"],[0.463,5,"M",0,0,"2026-10-21"],[0.463,7,"M",0,0,"2026-10-22"],[0.463,0,"C",0,0,"2026-10-22"],[0.463,0,"C",0,0,"2026-10-23"],[0.463,0,"C",1,0,"2026-10-24"],[0.463,0,"C",1,0,"2026-10-26"],[0.463,0,"C",1,1,"2026-10-26"],[0.463,0,"E",0,0,"2026-10-27"],[0.462,1,"C",1,0,"2026-10-26"],[0.462,0,"C",1,0,"2026-10-27"],[0.461,0,"C",1,0,"2026-10-26"],[0.461,0,"C",1,0,"2026-10-27"],[0.46,1,"C",0,0,"2026-10-19"],[0.46,0,"C",1,0,"2026-10-24"],[0.46,0,"C",1,0,"2026-10-27"],[0.459,1,"C",0,0,"2026-10-21"],[0.459,0,"C",1,0,"2026-10-22"],[0.459,0,"C",1,0,"2026-10-26"],[0.459,0,"C",1,1,"2026-10-28"],[0.458,0,"C",1,0,"2026-10-23"],[0.457,0,"C",0,0,"2026-10-23"],[0.457,0,"C",1,0,"2026-10-24"],[0.457,0,"C",1,0,"2026-10-25"],[0.457,0,"C",1,0,"2026-10-26"],[0.456,155,"M",0,0,"2026-09-15"],[0.456,9,"E",0,0,"2026-10-17"],[0.456,77,"C",0,0,"2026-10-18"],[0.456,1,"C",0,0,"2026-10-19"],[0.456,2,"C",0,0,"2026-10-19"],[0.456,1,"E",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"E",1,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",1,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-22"],[0.456,0,"C",1,0,"2026-10-23"],[0.456,1,"C",1,0,"2026-10-23"],[0.456,0,"C",1,0,"2026-10-23"],[0.456,0,"C",1,1,"2026-10-27"],[0.456,0,"E",0,0,"2026-10-27"],[0.455,166,"M",1,0,"2026-09-15"],[0.455,2,"E",0,0,"2026-10-18"],[0.455,40,"M",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,8,"M",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,1,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,1,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",1,0,"2026-10-23"],[0.455,0,"C",1,0,"2026-10-23"],[0.455,0,"C",1,1,"2026-10-27"],[0.455,0,"E",0,0,"2026-10-27"],[0.454,37,"C",0,0,"2026-10-17"],[0.454,15,"C",0,0,"2026-10-17"],[0.454,2,"E",0,0,"2026-10-18"],[0.454,1,"C",0,0,"2026-10-18"],[0.454,0,"M",0,0,"2026-10-19"],[0.454,0,"C",0,0,"2026-10-19"],[0.454,0,"C",0,0,"2026-10-20"],[0.454,0,"E",0,0,"2026-10-20"],[0.454,0,"E",0,0,"2026-10-20"],[0.454,0,"M",0,0,"2026-10-22"],[0.454,1,"C",1,0,"2026-10-24"],[0.452,36,"E",0,0,"2026-10-14"],[0.452,0,"E",0,0,"2026-10-27"],[0.451,8,"E",0,0,"2026-10-02"],[0.451,1,"C",0,0,"2026-10-12"],[0.451,73,"E",0,0,"2026-10-15"],[0.451,48,"E",0,0,"2026-10-15"],[0.451,14,"E",0,0,"2026-10-16"],[0.451,2,"C",0,0,"2026-10-16"],[0.451,2,"C",0,0,"2026-10-16"],[0.451,3,"C",0,0,"2026-10-17"],[0.451,0,"E",0,0,"2026-10-17"],[0.451,4,"M",0,0,"2026-10-17"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-19"],[0.45,0,"C",0,0,"2026-10-22"],[0.448,2,"E",1,0,"2026-10-17"],[0.448,2,"E",0,0,"2026-10-18"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,2,"M",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-20"],[0.448,0,"E",0,0,"2026-10-25"],[0.448,1,"E",0,0,"2026-10-25"],[0.448,0,"C",1,1,"2026-10-27"],[0.447,0,"C",0,0,"2026-10-17"],[0.447,0,"C",0,0,"2026-10-18"],[0.447,1,"M",0,0,"2026-10-20"],[0.447,0,"E",0,0,"2026-10-24"],[0.447,0,"C",1,1,"2026-10-25"],[0.447,0,"C",1,1,"2026-10-25"],[0.447,0,"C",1,1,"2026-10-25"],[0.446,0,"C",0,0,"2026-10-14"],[0.446,0,"C",0,0,"2026-10-14"],[0.446,63,"E",0,0,"2026-10-14"],[0.446,7,"M",0,0,"2026-10-15"],[0.446,2,"E",0,0,"2026-10-15"],[0.446,1,"C",0,0,"2026-10-15"],[0.446,1,"M",0,0,"2026-10-17"],[0.446,0,"E",0,0,"2026-10-20"],[0.446,0,"C",0,0,"2026-10-22"],[0.445,0,"C",0,0,"2026-10-15"],[0.445,0,"C",0,0,"2026-10-17"],[0.445,0,"C",0,0,"2026-10-20"],[0.445,0,"C",0,0,"2026-10-20"],[0.444,2,"C",0,0,"2026-10-13"],[0.444,0,"C",0,0,"2026-10-15"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-20"],[0.444,0,"C",0,0,"2026-10-20"],[0.443,18,"C",0,0,"2026-10-13"],[0.443,0,"C",0,0,"2026-10-15"],[0.443,3,"C",0,0,"2026-10-15"],[0.443,0,"C",0,0,"2026-10-20"],[0.442,25,"C",0,0,"2026-10-13"],[0.442,1,"C",0,0,"2026-10-17"],[0.442,0,"C",0,0,"2026-10-22"],[0.441,62,"E",0,0,"2026-10-10"],[0.441,69,"M",0,0,"2026-10-12"],[0.441,12,"E",0,0,"2026-10-13"],[0.441,2,"C",0,0,"2026-10-13"],[0.441,3,"C",0,0,"2026-10-13"],[0.441,0,"E",0,0,"2026-10-14"],[0.441,0,"C",0,0,"2026-10-14"],[0.441,1,"C",0,0,"2026-10-15"],[0.44,139,"M",0,0,"2026-09-17"],[0.44,3,"E",0,0,"2026-09-28"],[0.44,125,"M",0,0,"2026-10-10"],[0.44,3,"C",0,0,"2026-10-10"],[0.44,19,"E",0,0,"2026-10-11"],[0.44,26,"E",0,0,"2026-10-11"],[0.44,0,"E",0,0,"2026-10-11"],[0.44,2,"C",0,0,"2026-10-12"],[0.44,1,"C",0,0,"2026-10-12"],[0.44,0,"C",0,0,"2026-10-12"],[0.44,1,"M",0,0,"2026-10-12"],[0.44,1,"E",0,0,"2026-10-12"],[0.44,0,"C",0,0,"2026-10-13"],[0.44,1,"E",0,0,"2026-10-21"],[0.439,1,"M",0,0,"2026-10-17"],[0.438,2,"C",0,0,"2026-10-09"],[0.438,59,"C",0,0,"2026-10-09"],[0.438,0,"C",0,0,"2026-10-11"],[0.438,0,"C",0,0,"2026-10-15"],[0.438,0,"C",0,0,"2026-10-15"],[0.438,2,"C",0,0,"2026-10-16"],[0.438,0,"C",0,0,"2026-10-18"],[0.438,0,"E",0,0,"2026-10-21"],[0.438,0,"C",0,0,"2026-10-21"],[0.436,1,"M",0,0,"2026-10-14"],[0.436,0,"C",0,0,"2026-10-14"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-17"],[0.436,1,"C",0,0,"2026-10-17"],[0.436,0,"C",0,0,"2026-10-18"],[0.436,0,"C",0,0,"2026-10-18"],[0.436,0,"E",0,0,"2026-10-25"],[0.435,25,"E",0,0,"2026-10-09"],[0.435,0,"C",0,0,"2026-10-15"],[0.435,0,"C",0,0,"2026-10-16"],[0.435,0,"C",0,0,"2026-10-20"],[0.435,0,"C",1,0,"2026-10-26"],[0.434,0,"C",0,0,"2026-10-12"],[0.434,0,"C",0,0,"2026-10-16"],[0.434,0,"C",0,0,"2026-10-16"],[0.434,0,"C",0,0,"2026-10-22"],[0.433,42,"E",0,0,"2026-09-17"],[0.433,29,"E",0,0,"2026-10-07"],[0.433,5,"C",0,0,"2026-10-07"],[0.433,48,"C",0,0,"2026-10-08"],[0.433,8,"C",0,0,"2026-10-09"],[0.433,0,"C",0,0,"2026-10-12"],[0.433,0,"M",1,0,"2026-10-13"],[0.433,0,"C",0,0,"2026-10-14"],[0.431,2,"M",0,0,"2026-10-14"],[0.431,0,"C",0,0,"2026-10-14"],[0.431,2,"M",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"M",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-21"],[0.43,0,"C",0,0,"2026-10-12"],[0.43,1,"C",0,0,"2026-10-12"],[0.429,0,"C",0,0,"2026-10-14"],[0.429,0,"E",0,0,"2026-10-25"],[0.428,86,"C",0,0,"2026-10-07"],[0.428,0,"C",0,0,"2026-10-11"],[0.426,2,"C",0,0,"2026-10-13"],[0.425,107,"C",0,0,"2026-10-05"],[0.425,7,"C",0,0,"2026-10-06"],[0.425,23,"E",0,0,"2026-10-06"],[0.425,9,"C",0,0,"2026-10-07"],[0.425,0,"C",0,0,"2026-10-20"],[0.425,0,"C",0,0,"2026-10-21"],[0.424,3,"C",0,0,"2026-10-07"],[0.424,2,"C",0,0,"2026-10-07"],[0.423,1,"E",0,0,"2026-10-07"],[0.423,3,"M",0,0,"2026-10-08"],[0.423,0,"C",0,0,"2026-10-09"],[0.422,0,"C",0,0,"2026-10-09"],[0.422,0,"C",0,0,"2026-10-09"],[0.422,0,"E",0,0,"2026-10-24"],[0.421,56,"C",0,0,"2026-10-02"],[0.421,2,"C",0,0,"2026-10-06"],[0.421,0,"C",0,0,"2026-10-08"],[0.421,0,"C",0,0,"2026-10-08"],[0.42,16,"E",0,0,"2026-09-23"],[0.42,0,"C",0,0,"2026-10-07"],[0.42,0,"C",0,0,"2026-10-18"],[0.419,0,"C",0,0,"2026-10-06"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"M",0,0,"2026-10-20"],[0.419,0,"C",0,0,"2026-10-21"],[0.419,0,"E",0,0,"2026-10-27"],[0.418,0,"C",0,0,"2026-10-07"],[0.418,0,"C",0,0,"2026-10-08"],[0.418,0,"C",0,0,"2026-10-09"],[0.418,3,"C",0,0,"2026-10-09"],[0.418,0,"E",0,0,"2026-10-09"],[0.418,0,"C",0,0,"2026-10-10"],[0.417,79,"C",0,0,"2026-10-03"],[0.417,6,"E",0,0,"2026-10-03"],[0.417,1,"E",0,0,"2026-10-05"],[0.417,0,"C",0,0,"2026-10-06"],[0.417,0,"E",0,0,"2026-10-06"],[0.417,0,"C",0,0,"2026-10-08"],[0.416,0,"E",0,0,"2026-10-06"],[0.415,93,"C",0,0,"2026-10-01"],[0.415,0,"C",0,0,"2026-10-07"],[0.415,0,"C",0,0,"2026-10-07"],[0.415,0,"C",0,0,"2026-10-08"],[0.415,0,"C",0,0,"2026-10-09"],[0.415,0,"C",1,0,"2026-10-26"],[0.414,0,"C",0,0,"2026-10-07"],[0.413,5,"C",0,0,"2026-10-01"],[0.413,33,"C",0,0,"2026-10-03"],[0.413,2,"C",0,0,"2026-10-04"],[0.413,9,"C",0,0,"2026-10-04"],[0.413,0,"C",0,0,"2026-10-04"],[0.413,8,"C",0,0,"2026-10-05"],[0.413,0,"C",0,0,"2026-10-08"],[0.411,2,"C",0,0,"2026-10-01"],[0.411,2,"C",0,0,"2026-10-03"],[0.411,0,"C",0,0,"2026-10-05"],[0.411,0,"C",0,0,"2026-10-08"],[0.409,1,"C",0,0,"2026-10-01"],[0.409,27,"C",0,0,"2026-10-01"],[0.409,4,"C",0,0,"2026-10-02"],[0.409,2,"C",0,0,"2026-10-02"],[0.409,1,"C",0,0,"2026-10-03"],[0.409,0,"C",0,0,"2026-10-05"],[0.409,0,"E",0,0,"2026-10-06"],[0.409,0,"C",0,0,"2026-10-08"],[0.407,1,"C",0,0,"2026-10-02"],[0.407,0,"C",0,0,"2026-10-04"],[0.407,0,"C",0,0,"2026-10-05"],[0.406,1,"C",0,0,"2026-10-04"],[0.406,0,"C",0,0,"2026-10-04"],[0.406,0,"C",0,0,"2026-10-06"],[0.405,30,"E",0,0,"2026-09-28"],[0.405,0,"C",0,0,"2026-10-04"],[0.404,2,"E",0,0,"2026-10-02"],[0.404,0,"E",0,0,"2026-10-06"],[0.404,0,"C",1,1,"2026-10-24"],[0.403,1,"C",0,0,"2026-10-05"],[0.402,0,"C",0,0,"2026-10-02"],[0.402,0,"E",0,0,"2026-10-05"],[0.401,32,"C",0,0,"2026-09-30"],[0.4,1,"C",0,0,"2026-10-01"],[0.4,2,"E",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-16"],[0.4,0,"C",0,0,"2026-10-16"],[0.4,0,"E",0,0,"2026-10-24"],[0.399,56,"E",0,0,"2026-09-30"],[0.399,5,"M",0,0,"2026-09-30"],[0.399,1,"C",0,0,"2026-10-01"],[0.399,0,"C",0,0,"2026-10-01"],[0.399,1,"C",0,0,"2026-10-02"],[0.399,0,"C",0,0,"2026-10-06"],[0.398,1,"C",0,0,"2026-10-12"],[0.397,0,"E",0,0,"2026-10-01"],[0.397,0,"C",0,0,"2026-10-01"],[0.397,1,"C",0,0,"2026-10-02"],[0.396,0,"E",0,0,"2026-10-01"],[0.396,0,"E",0,0,"2026-10-02"],[0.396,0,"C",0,0,"2026-10-02"],[0.395,0,"C",1,0,"2026-10-01"],[0.394,0,"C",0,0,"2026-10-06"],[0.393,1,"C",0,0,"2026-10-02"],[0.393,1,"C",0,0,"2026-10-02"],[0.393,0,"E",0,0,"2026-10-03"],[0.393,0,"M",0,0,"2026-10-03"],[0.393,0,"E",0,0,"2026-10-04"],[0.393,0,"E",0,0,"2026-10-04"],[0.39,0,"C",0,0,"2026-09-30"],[0.389,0,"C",0,0,"2026-10-04"],[0.386,17,"E",0,0,"2026-09-29"],[0.386,0,"M",0,0,"2026-10-02"],[0.386,0,"M",0,0,"2026-10-02"],[0.386,0,"C",0,0,"2026-10-02"],[0.386,0,"C",1,0,"2026-10-09"],[0.384,23,"M",0,0,"2026-09-29"],[0.384,5,"C",0,0,"2026-09-29"],[0.384,1,"C",0,0,"2026-09-30"],[0.384,0,"C",0,0,"2026-09-30"],[0.384,1,"C",0,0,"2026-09-30"],[0.384,0,"C",0,0,"2026-10-01"],[0.384,0,"M",0,0,"2026-10-18"],[0.384,0,"E",0,0,"2026-10-25"],[0.383,0,"C",0,0,"2026-10-02"],[0.383,0,"M",0,0,"2026-10-17"],[0.382,0,"C",0,0,"2026-09-29"],[0.382,0,"C",1,0,"2026-10-02"],[0.382,0,"C",0,0,"2026-10-06"],[0.381,29,"M",0,0,"2026-09-29"],[0.381,0,"C",0,0,"2026-10-02"],[0.381,0,"M",0,0,"2026-10-19"],[0.38,0,"C",0,0,"2026-09-29"],[0.38,1,"C",0,0,"2026-10-11"],[0.379,0,"C",0,0,"2026-09-29"],[0.378,24,"M",0,0,"2026-09-29"],[0.378,1,"C",0,0,"2026-10-01"],[0.378,0,"E",0,0,"2026-10-03"],[0.378,0,"E",0,0,"2026-10-04"],[0.377,0,"C",0,0,"2026-09-30"],[0.377,0,"C",0,0,"2026-10-02"],[0.377,0,"E",0,0,"2026-10-04"],[0.375,0,"C",0,0,"2026-10-02"],[0.375,0,"C",0,0,"2026-10-03"],[0.375,1,"C",0,0,"2026-10-04"],[0.375,0,"E",0,0,"2026-10-05"],[0.374,2,"E",1,0,"2026-09-30"],[0.374,0,"C",0,0,"2026-10-03"],[0.374,0,"E",0,0,"2026-10-05"],[0.373,19,"M",0,0,"2026-09-28"],[0.372,0,"C",0,0,"2026-10-04"],[0.371,0,"C",0,0,"2026-10-06"],[0.37,23,"M",0,0,"2026-09-26"],[0.37,0,"C",0,0,"2026-09-29"],[0.37,0,"C",0,0,"2026-10-05"],[0.368,0,"C",0,0,"2026-09-29"],[0.367,0,"C",0,0,"2026-09-29"],[0.366,0,"C",0,0,"2026-09-28"],[0.365,0,"C",0,0,"2026-09-30"],[0.364,0,"C",1,0,"2026-10-03"],[0.362,20,"M",0,0,"2026-09-26"],[0.359,0,"C",1,1,"2026-10-26"],[0.359,0,"C",1,1,"2026-10-27"],[0.358,30,"M",0,0,"2026-09-21"],[0.356,0,"C",0,0,"2026-09-29"],[0.356,0,"C",0,0,"2026-09-30"],[0.354,45,"M",0,0,"2026-09-21"],[0.348,0,"C",0,0,"2026-09-28"],[0.347,42,"C",0,0,"2026-09-17"],[0.345,0,"C",0,0,"2026-09-28"],[0.344,0,"C",0,0,"2026-09-28"],[0.344,1,"C",0,0,"2026-10-24"],[0.342,16,"M",0,0,"2026-09-18"],[0.341,40,"E",0,0,"2026-09-20"],[0.341,2,"E",0,0,"2026-09-21"],[0.341,2,"C",0,0,"2026-09-22"],[0.341,3,"C",0,0,"2026-09-25"],[0.341,5,"E",0,0,"2026-09-25"],[0.341,1,"C",0,0,"2026-09-25"],[0.341,0,"C",0,0,"2026-09-27"],[0.341,0,"C",0,0,"2026-09-27"],[0.341,7,"E",0,0,"2026-09-29"],[0.339,270,"E",0,0,"2026-09-15"],[0.339,0,"C",0,0,"2026-09-25"],[0.339,1,"C",0,0,"2026-09-26"],[0.339,0,"C",0,0,"2026-10-24"],[0.338,0,"C",0,0,"2026-09-21"],[0.338,0,"C",0,0,"2026-09-26"],[0.337,81,"E",0,0,"2026-09-16"],[0.337,33,"E",0,0,"2026-09-17"],[0.337,0,"E",0,0,"2026-09-20"],[0.337,0,"C",0,0,"2026-09-25"],[0.337,0,"C",0,0,"2026-09-25"],[0.335,0,"E",0,0,"2026-09-20"],[0.334,13,"E",0,0,"2026-09-19"],[0.334,0,"C",0,0,"2026-09-21"],[0.334,0,"C",0,0,"2026-09-21"],[0.334,0,"E",0,0,"2026-09-22"],[0.334,0,"C",0,0,"2026-09-25"],[0.334,0,"C",0,0,"2026-09-25"],[0.334,0,"C",0,0,"2026-10-24"],[0.333,3,"E",0,0,"2026-09-19"],[0.333,0,"E",0,0,"2026-09-20"],[0.333,0,"C",0,0,"2026-09-24"],[0.333,0,"E",0,0,"2026-09-25"],[0.333,3,"E",0,0,"2026-09-25"],[0.333,0,"C",0,0,"2026-09-27"],[0.333,0,"C",0,0,"2026-10-24"],[0.332,2,"E",0,0,"2026-09-20"],[0.332,0,"C",0,0,"2026-09-26"],[0.332,0,"C",0,0,"2026-09-26"],[0.331,3,"C",0,0,"2026-09-18"],[0.331,0,"E",0,0,"2026-09-20"],[0.331,0,"E",0,0,"2026-09-20"],[0.33,0,"C",0,0,"2026-09-20"],[0.33,1,"C",0,0,"2026-09-21"],[0.33,1,"C",0,0,"2026-09-24"],[0.329,0,"C",0,0,"2026-10-24"],[0.328,2,"E",0,0,"2026-09-17"],[0.328,1,"E",0,0,"2026-09-17"],[0.328,0,"C",0,0,"2026-09-18"],[0.328,0,"C",0,0,"2026-09-20"],[0.328,0,"C",0,0,"2026-09-21"],[0.327,0,"C",0,0,"2026-09-21"],[0.327,0,"C",0,0,"2026-09-23"],[0.326,0,"E",0,0,"2026-09-19"],[0.326,0,"E",0,0,"2026-09-19"],[0.326,0,"E",0,0,"2026-09-19"],[0.323,1,"E",0,0,"2026-09-18"],[0.323,2,"E",0,0,"2026-09-19"],[0.323,0,"C",0,0,"2026-09-21"],[0.322,1,"E",0,0,"2026-09-17"],[0.322,0,"C",0,0,"2026-09-25"],[0.32,1,"E",0,0,"2026-09-22"],[0.319,1,"C",0,0,"2026-09-18"],[0.319,1,"E",0,0,"2026-09-19"],[0.317,1,"C",0,0,"2026-09-20"],[0.315,0,"C",0,0,"2026-09-26"],[0.314,0,"C",0,0,"2026-09-25"],[0.311,0,"C",0,0,"2026-09-28"],[0.309,0,"C",0,0,"2026-09-26"],[0.308,1,"E",0,0,"2026-09-21"],[0.308,1,"E",0,0,"2026-09-21"],[0.301,2,"C",0,0,"2026-09-18"],[0.298,0,"C",0,0,"2026-09-25"],[0.297,1,"E",0,0,"2026-09-19"],[0.293,0,"C",0,0,"2026-09-17"],[0.293,1,"E",0,0,"2026-09-19"],[0.291,0,"C",0,0,"2026-10-11"],[0.29,1,"E",0,0,"2026-09-19"],[0.29,1,"E",0,0,"2026-09-19"],[0.288,0,"C",0,0,"2026-10-11"],[0.285,32,"M",0,0,"2026-09-15"],[0.285,1,"E",0,0,"2026-09-20"],[0.285,1,"E",0,0,"2026-09-20"],[0.285,0,"C",0,0,"2026-09-25"],[0.274,0,"C",0,0,"2026-10-26"],[0.274,0,"C",0,0,"2026-10-26"],[0.273,0,"C",0,0,"2026-10-26"],[0.272,2,"C",0,0,"2026-10-14"],[0.269,2,"E",0,0,"2026-09-15"],[0.264,0,"C",0,0,"2026-09-26"],[0.261,0,"C",0,0,"2026-10-03"],[0.255,0,"C",0,0,"2026-09-21"],[0.249,12,"E",0,0,"2026-09-17"],[0.241,8,"E",0,0,"2026-09-15"],[0.241,3,"C",0,0,"2026-09-16"],[0.239,0,"C",0,0,"2026-10-26"],[0.237,0,"C",0,0,"2026-10-22"],[0.236,0,"C",0,0,"2026-09-19"],[0.236,2,"C",0,0,"2026-09-19"],[0.236,0,"C",0,0,"2026-09-21"],[0.233,3,"M",0,0,"2026-09-15"],[0.231,0,"C",0,0,"2026-10-12"],[0.231,0,"C",0,0,"2026-10-12"],[0.231,0,"C",0,0,"2026-10-26"],[0.228,0,"E",0,0,"2026-10-14"],[0.227,4,"C",0,0,"2026-09-16"],[0.226,1,"C",0,0,"2026-10-12"],[0.224,1,"C",0,0,"2026-09-19"],[0.221,2,"C",0,0,"2026-10-11"],[0.217,0,"C",0,0,"2026-10-25"],[0.21,0,"C",0,0,"2026-10-25"],[0.207,0,"C",0,0,"2026-10-11"],[0.204,0,"C",0,0,"2026-10-26"],[0.191,1,"E",0,0,"2026-09-15"],[0.185,0,"C",0,0,"2026-10-17"],[0.182,2,"C",0,0,"2026-09-18"],[0.18,3,"E",0,0,"2026-09-15"],[0.18,2,"C",0,0,"2026-09-21"],[0.179,0,"C",0,0,"2026-10-25"],[0.176,2,"C",0,0,"2026-10-11"],[0.174,3,"C",0,0,"2026-09-16"],[0.174,0,"E",0,0,"2026-09-17"],[0.171,0,"C",0,0,"2026-10-24"],[0.169,1,"E",0,0,"2026-09-18"],[0.166,1,"E",0,0,"2026-09-15"],[0.166,1,"E",0,0,"2026-09-15"],[0.166,1,"E",0,0,"2026-09-15"],[0.165,2,"E",0,0,"2026-09-27"],[0.163,1,"E",0,0,"2026-09-15"],[0.161,0,"C",0,0,"2026-10-25"],[0.16,8,"M",0,0,"2026-09-15"],[0.158,24,"E",0,0,"2026-09-15"],[0.157,8,"E",0,0,"2026-09-15"],[0.156,32,"E",0,0,"2026-09-15"],[0.156,0,"C",0,0,"2026-09-18"],[0.156,0,"E",0,0,"2026-09-21"],[0.156,0,"C",0,0,"2026-10-26"],[0.155,4,"M",0,0,"2026-09-15"],[0.153,2,"E",0,0,"2026-09-15"],[0.153,0,"C",0,0,"2026-09-24"],[0.153,8,"C",0,0,"2026-10-13"],[0.152,0,"C",0,0,"2026-09-17"],[0.151,2,"M",0,0,"2026-09-15"],[0.151,10,"E",0,0,"2026-09-20"],[0.151,9,"E",0,0,"2026-09-20"],[0.15,2,"C",0,0,"2026-10-11"],[0.149,11,"E",0,0,"2026-09-15"],[0.149,2,"C",0,0,"2026-09-16"],[0.149,0,"C",0,0,"2026-10-13"],[0.146,0,"E",0,0,"2026-09-15"],[0.146,1,"E",0,0,"2026-09-15"],[0.145,0,"C",0,0,"2026-09-25"],[0.145,0,"C",0,0,"2026-10-09"],[0.143,3,"E",0,0,"2026-09-15"],[0.142,6,"C",0,0,"2026-09-16"],[0.142,1,"C",0,0,"2026-09-16"],[0.14,24,"E",0,0,"2026-09-15"],[0.14,0,"C",0,0,"2026-10-15"],[0.136,0,"C",0,0,"2026-10-04"],[0.134,2,"C",0,0,"2026-09-21"],[0.134,0,"C",0,0,"2026-10-01"],[0.13,0,"C",0,0,"2026-09-22"],[0.128,0,"E",0,0,"2026-09-15"],[0.115,0,"C",0,0,"2026-10-01"],[0.11,2,"E",0,0,"2026-09-17"],[0.106,0,"C",0,0,"2026-09-20"],[0.103,0,"C",0,0,"2026-09-18"],[0.099,0,"M",0,0,"2026-10-26"],[0.098,0,"C",0,0,"2026-10-04"],[0.096,3,"E",0,0,"2026-09-15"],[0.094,0,"C",0,0,"2026-09-24"],[0.093,0,"C",0,0,"2026-10-13"],[0.09,0,"C",0,0,"2026-09-19"],[0.09,0,"C",0,0,"2026-10-01"],[0.089,26,"E",0,0,"2026-09-21"],[0.088,0,"C",0,0,"2026-09-22"],[0.086,0,"C",0,0,"2026-10-23"],[0.083,0,"E",0,0,"2026-09-17"],[0.082,0,"E",0,0,"2026-09-29"],[0.064,0,"E",0,0,"2026-09-24"],[0.05,0,"C",0,0,"2026-10-18"],[0.046,0,"C",0,0,"2026-10-24"],[0.038,9,"E",0,0,"2026-09-20"],[0.0,419,"S",0,0,"2026-09-14"],[0.0,0,"C",0,0,"2026-09-17"],[0.0,0,"C",0,0,"2026-09-18"],[0.0,0,"E",0,0,"2026-09-18"],[0.0,0,"C",0,0,"2026-09-18"],[0.0,0,"C",0,0,"2026-09-23"],[0.0,0,"C",0,0,"2026-09-25"],[0.0,0,"C",0,0,"2026-09-25"],[0.0,3,"E",0,0,"2026-09-27"],[0.0,0,"C",0,0,"2026-10-22"],[0.0,0,"C",0,0,"2026-10-22"],[0.0,0,"C",1,1,"2026-10-26"]]}
if SHOW:
    import matplotlib.pyplot as plt
    style_apply()
    sc = np.array([r[0] for r in FIELD['scored']])
    v4g = np.array([r[3] for r in FIELD['scored']], bool)
    lo = 0.40
    fig, ax = plt.subplots(figsize=(10, 4.4))
    bins = np.arange(lo - 0.0005, 0.4806, 0.001)   # one bar per printed score (three decimals)
    ax.hist([sc[(sc >= lo) & ~v4g], sc[(sc >= lo) & v4g]], bins=bins, stacked=True, color=['#ced4da', '#495057'],
            label=['other notebooks', "attach Dmitrii Gluzdov's v4g assets"])
    top = ax.get_ylim()[1]
    for x, name in [(0.465, '0.465 frozen pipeline'), (0.468, '0.468 leader-safe V2'), (0.472, '0.472 two ways'), (0.477, 'this notebook 0.477')]:
        ours = name.startswith('this')
        ax.axvline(x, color=ACCENT if ours else MUTED, lw=1.4 if ours else 1.0, ls='-' if ours else ':')
        ax.text(x - 0.0006, top * 0.98, name, rotation=90, va='top', ha='right', fontsize=8.5,
                color=ACCENT if ours else INK, bbox=dict(facecolor='white', edgecolor='none', pad=1.0, alpha=0.85))
    ax.set_xlim(lo, 0.48)
    ax.set_xlabel('public LB shown on the notebook page'); ax.set_ylabel('notebooks')
    ax.set_title(f"{int((sc >= 0.46).sum())} notebooks at 0.460 or more, {int(v4g[sc >= 0.46].sum())} of them on one set of assets")
    ax.legend(loc='upper left')
    print(f"{FIELD['notebooks_listed']} public notebooks, {len(sc)} with a score, {int((sc < lo).sum())} of those below {lo} not drawn")
    plt.tight_layout(); plt.show()

*Figure 1. Public scores as the Code tab shows them (the best over each notebook's versions), 0.40 and above. Dark:
notebooks that attach Dmitrii Gluzdov's v4g assets.*

## 2. One lineage, two branches

| step | notebook | what it changed | public LB shown |
|---|---|---|---:|
| 1 | Dmitrii Gluzdov, [The Data Behind Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures) | assembles and documents the shared pipeline: library search, a PubChem channel, GLACIER and ICEBERG forward models, two-engine fusion (its own credits name canqiang's E006 and Ahmed Berat Özer's engines as the base) | 0.465 |
| 2 | Saner Turhaner, [PubChem + Neural025](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466) | a neural spectrum-to-structure scorer, fused by reciprocal rank at weight 0.25 | 0.466 |
| 3 | Roman Rozen, [From Peaks to Structures](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures) | leader-safe consensus V2: rank 1 as in step 2, ranks 2-25 at neural weight 0.50 (Kozykorpesh's [Balanced Acquisition Evidence](https://www.kaggle.com/code/kozykappa/lb-0-468-balanced-acquisition-evidence) ran this V2 in its versions 6-7, the pipeline Spectral Parallax thanks) | 0.468 |
| 4a | alex chilton, [Leader-Safe, Ranks 2-25 at 0.75](https://www.kaggle.com/code/alexchilton/casmi26-lb-0-472-leader-safe-ranks-2-25-at-0-75) | ranks 2-25 at neural weight 0.75 | 0.472 on [a fork that changes only the commit mode](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472); his own page shows 0.467 |
| 4b | Roman Tamrazov, [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax) | an untrained fragment check on the V2 lists | 0.472 (best of its versions; version 22, whose code runs here, shows no score of its own) |
| 5 | this notebook | 4b's leader moves from the V2 order, ranks 2-25 from 4a's 0.75 order | **0.477** |

Scores are what the Code tab displayed on 2026-10-27, Kaggle's best public score over each notebook's versions. The
current versions of some linked notebooks run something else: Roman Rozen's now stacks a private isomer reranker
(skipped when its dataset is absent), the fragment check and a tail recovery on ranks 2-25 at 1.00, and since
2026-10-28 his page shows 0.472.

The fusion rule in one line: a candidate at pipeline rank b and neural rank n scores 1/(3 + b) + w/(3 + n). Rank 1
uses w = 0.25, so the neural model can take the lead from the pipeline's leader only when that leader sits at neural
rank 18 or worse and the runner-up is the neural favourite. Ranks 2-25 use w_mid; at 0.75 the neural model sorts them
much harder than at 0.50.

**What the fragment check does.** 4b has three parts:

- **Rank 1.** When the leader's match to the reference library is below 0.87 and the molecule has at least three
  usable spectra, a candidate from ranks 2-6 with the leader's formula (an isomer the precursor mass cannot tell
  apart) can take rank 1. It must explain clearly more fragment intensity than the leader and score higher with the
  neural model, and it needs support from the engines' own top-5 lists or from the GLACIER and ICEBERG forward
  models.
- **Swaps.** Up to two adjacent swaps of same-formula candidates in ranks 2-11, with no library gate.
- **Last place.** One same-formula outside candidate may take rank 25.

**Why stacking 4a and 4b scored 0.472, and what scored 0.477.** 4a never touches rank 1, so the two changes looked
independent. Run together (our e1) they scored 0.472, the same as each alone. The rank-1 rule takes its contenders
from ranks 2-6 of the order it is handed. On the 0.75 order that window holds different candidates: on Saner
Turhaner's labelled panel, the same-formula candidates in ranks 2-6 that pass the rule's neural gates change in 17 of
86 eligible lists, each change adds a wrong structure and no right one drops out (the fragment gates need the
spectra and were not replayed). Our reading, which no single submission isolates: on the 0.75 order the rule keeps
its good leader moves and adds wrong ones. The whole fragment check (rank-1 rule, swaps and last place) on the
long-trained neural model's 0.75 order cost 0.008 (e3 0.453 against e4 0.461); that pair does not separate the rank-1
rule from the other two parts. This
notebook runs the rule on the V2 order, keeps only the leader it picks there, and places the old leader second and
the 0.75 order below them: 0.477.

The field already swept the knobs around 4a (displayed scores, knobs read from each notebook's configuration cell):
w_mid 0.70, 0.75 and 0.80 all show 0.472 ([v1220](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1220-golden-ratio-070), [fork-lb472](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472),
[v1210](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1210-sweet-spot-080)); rank-1 weight 0 shows 0.472 ([v1230](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1230-pure-shield-075)), 0.50 shows 0.448 and 1.00 shows
0.359 ([fork-wt05](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt05), [fork-wt10](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt10)). On the step-3 line (w_mid 0.50), offset 1 instead
of 3 shows 0.466 ([v520](https://www.kaggle.com/code/guhongbin/casmi26-v520-p2s-wmid050off1)). Moving rank 1 with the neural model is where scores are lost.

## 3. The pipeline

The configuration and session cells at the top of this notebook and the cells below are Spectral Parallax's, in his
order. Hidden: the pinned sources (Saner Turhaner's six helper modules from the 0.466 notebooks, Roman Rozen's
`consensus.py` and `outsiders.py`, and the 0.465 pipeline embedded as base64, each checked by SHA-256), the stage
runner, and the fragment check's source. Shown: the run cell (pipeline in one process, the neural scorer on the final
lists and on up to 20 outside candidates per molecule, the leader-safe consensus), the fragment-check cell and the
provenance cell. A failing overlay restores the previous answer file; the scoring rerun refuses to submit without a
real pipeline run and a neural score.

In [ ]:
# ── Pinned sources (generated; hidden in the viewer) ─────────────────────────────────────────
# MODULES: the six helper modules of the public 0.466 notebooks, unchanged, plus this notebook's
# consensus.py and outsiders.py. FROZEN_B64: the public 0.465 pipeline notebook, byte for byte.
MODULES = {}
MODULES['runner.py'] = '"""Execute the pinned corrected baseline once, then export its final candidates."""\nimport os,sys,json,hashlib\nfrom pathlib import Path\nfrom IPython.display import display\nif __name__==\'__main__\':\n    source=Path(sys.argv[1]); expected=sys.argv[2]; root=Path(sys.argv[3]).resolve()\n    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected, \'Source hash differs\'\n    assert not os.environ.get(\'INC_PANEL\'), \'Replay input is not allowed\'\n    root.mkdir(parents=True,exist_ok=True); os.chdir(root)\n    assert not (root/\'work/spec_meta.parquet\').exists(), \'Fresh pipeline directory required\'\n    assert not (root/\'submission.csv\').exists(), \'Fresh pipeline directory required\'\n    scope={\'__name__\':\'__main__\',\'display\':display}\n    for i,c in enumerate(json.loads(source.read_text())[\'cells\']):\n        if c[\'cell_type\']!=\'code\': continue\n        print(f\'PIPELINE CELL {i} START\',flush=True)\n        exec(compile(\'\'.join(c[\'source\']),f\'pipeline_cell_{i}\',\'exec\'),scope)\n        print(f\'PIPELINE CELL {i} DONE\',flush=True)\n    assert \'candidate_promotion_evidence\' in scope, \'Candidate-specific promotion fix missing\'\n    from export_evidence import export\n    export(scope,root)\n    (root/\'execution_receipt.json\').write_text(json.dumps(dict(source_sha256=expected,\n        mode=\'submission\',source_adaptations=[],candidate_promotion_fix=True,\n        is_full_run=bool(scope[\'IS_RERUN\']),processed_molecules=int(scope[\'te\'].molecule_id.nunique())),indent=2))\n'
MODULES['export_evidence.py'] = '"""Executed in the frozen pipeline process AFTER every original final rule."""\nimport math,json,shutil\nfrom pathlib import Path\nimport pandas as pd\n\ndef export(scope, root):\n    root=Path(root);frame=pd.read_csv(root/\'submission.csv\',dtype=str,keep_default_na=False)\n    shutil.copy2(root/\'submission.csv\',root/\'submission_baseline0465.csv\')\n    processed=set(scope[\'te\'].molecule_id.astype(str));identity=scope[\'metric_identity\']\n    def rank_map(smis):\n        out={}\n        for rank,s in enumerate(smis,1):\n            key=identity(s)\n            if key is not None:out.setdefault(key,rank)\n        return out\n    def finite(v):\n        try:return float(v) if v is not None and math.isfinite(float(v)) else None\n        except (ValueError,TypeError):return None\n    rows=[]\n    for mid,answer in frame.itertuples(index=False):\n        candidates=answer.split(\';\')\n        assert 1<=len(candidates)<=25 and len(candidates)==len(set(candidates)),mid\n        active=mid in processed\n        base=scope[\'BASE\'].get(mid,([],[],0.))\n        maps={\n            \'main\':rank_map(base[0]),\n            \'prefusion\':rank_map(scope.get(\'PREFUSION_LISTS\',{}).get(mid,[])),\n            \'engine2\':rank_map(scope.get(\'ENG\',{}).get(mid,{}).get(\'smiles\',[])),\n            \'pubchem\':rank_map(scope.get(\'PC\',{}).get(mid,{}).get(\'pc\',[]))}\n        ice=scope.get(\'ICE_SCORES\',{}).get(mid,{}) or {}\n        gl=scope[\'gl_of\'](mid) or {}\n        for rank,s in enumerate(candidates,1):\n            ident=identity(s);key=scope[\'chem\'].score_key(s)\n            row=dict(molecule_id=mid,smiles=s,metric_key=ident[0] if ident else None,baseline_rank=rank,\n                processed=active,library_max=finite(base[2]) or 0.,proxy_gl=mid in scope.get(\'N1_GL_MIDS\',set()),\n                ice=finite(ice.get(s,ice.get(key))),gl=finite(gl.get(s,gl.get(key))))\n            row.update({name+\'_rank\':mapping.get(ident,0) for name,mapping in maps.items()});rows.append(row)\n    pd.DataFrame(rows).to_parquet(root/\'frozen_candidate_evidence.parquet\',index=False)\n    # Save the effective query spectra used by the pipeline (12 rows\' molecules in visible smoke, all in rerun).\n    scope[\'te\'].to_parquet(root/\'effective_query_spectra.parquet\',index=False)\n    (root/\'frozen_run_info.json\').write_text(json.dumps(dict(processed_molecules=len(processed),\n        output_molecules=len(frame),is_full_run=bool(scope[\'IS_RERUN\']),\n        note=\'Baseline file saved after the original Engine2 rank-2 insertion; candidate-specific PubChem promotion fix included.\'),indent=2))\n    print(\'EXPORTED FROZEN FINAL25\',len(rows),\'candidate rows\',flush=True)\n'
MODULES['neural_scores.py'] = '"""Runs after the public pipeline process exits, freeing its GPU memory."""\nimport json,sys,importlib.util,types,hashlib\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\n\ndef load_model(bundle,checkpoint,device):\n    # Private module namespace prevents collision with public engines\' model/features modules.\n    package=\'_casmi_own\';p=types.ModuleType(package);p.__path__=[str(bundle)];sys.modules[package]=p\n    for name in [\'features\',\'model\']:\n        spec=importlib.util.spec_from_file_location(package+\'.\'+name,Path(bundle)/(name+\'.py\'))\n        mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod)\n    features=sys.modules[package+\'.features\'];cls=sys.modules[package+\'.model\'].SpectrumCandidateModel\n    cfg=json.loads((Path(bundle)/\'config.json\').read_text());model=cls(cfg).to(device)\n    ck=torch.load(Path(bundle)/checkpoint,map_location=device,weights_only=True)\n    model.load_state_dict(ck[\'state_dict\']);model.eval()\n    return model,features,cfg\n\n@torch.inference_mode()\ndef score(bundle,checkpoint,evidence,queries,output,device=\'cuda\'):\n    torch.set_num_threads(4)\n    if device==\'cuda\':assert torch.cuda.is_available(),\'GPU required for neural scoring\'\n    model,feat,cfg=load_model(bundle,checkpoint,device)\n    frame=pd.read_parquet(evidence);q=pd.read_parquet(queries);q[\'molecule_id\']=q.molecule_id.astype(str)\n    query={mid:g for mid,g in q.groupby(\'molecule_id\',sort=False)};cache={};parts=[];errors=[]\n    for index,(mid,g) in enumerate(frame.groupby(\'molecule_id\',sort=False)):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\').copy()\n        g[\'neural_ok\']=False\n        if not g.processed.iloc[0]:\n            for c in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[c]=0.\n            parts.append(g);continue\n        try:\n            assert mid in query,mid\n            records=[]\n            # Use EVERY observed query spectrum; no adduct merging and no new spectrum selection rule.\n            for row in query[mid].itertuples(index=False):\n                try:\n                    pk,n=feat.peaks(row.ms2_mzs,row.ms2_normalized_intensities,row.precursor_mz,cfg[\'max_peaks\'])\n                    if n<cfg[\'min_peaks\']:continue\n                    m,a=feat.metadata(row.precursor_mz,row.adduct,row.ionization_mode,row.collision_energy_ev)\n                    records.append((pk,n,m,a))\n                except (ValueError,TypeError):continue\n            if not records:raise ValueError(\'No usable spectra after the training preprocessing filters\')\n            fp=[]\n            for s in g.smiles:\n                if s not in cache:cache[s]=feat.structure(s,cfg[\'fp_bits\'])[-1]\n                fp.append(cache[s])\n            F=torch.tensor(np.stack(fp),device=device,dtype=torch.float32);M=model.encode_molecules(F)\n            cos_all=[];fp_all=[]\n            for st in range(0,len(records),64):\n                b=records[st:st+64]\n                x=torch.tensor(np.stack([v[0] for v in b]),device=device)\n                lengths=torch.tensor([v[1] for v in b],device=device,dtype=torch.long)\n                meta=torch.tensor(np.stack([v[2] for v in b]),device=device)\n                adduct=torch.tensor([v[3] for v in b],device=device,dtype=torch.long)\n                Z,P=model.encode_spectra(x,lengths,meta,adduct)\n                cos_all.append((Z@M.T).cpu().numpy());fp_all.append((P@F.T/cfg[\'fp_bits\']).cpu().numpy())\n            cos=np.concatenate(cos_all);fps=np.concatenate(fp_all)\n            combined=cos/cfg[\'temperature\']+cfg[\'fp_score_weight\']*fps\n            assert np.isfinite(combined).all(),mid\n            g[\'neural_cos\']=cos.mean(0);g[\'neural_fp\']=fps.mean(0);g[\'neural_score\']=combined.mean(0)\n            g[\'neural_std\']=combined.std(0);g[\'n_spectra\']=len(records);g[\'n_adducts\']=query[mid].adduct.nunique()\n            g[\'neural_ok\']=True\n            parts.append(g)\n        except (ValueError,TypeError) as exc:\n            errors.append(dict(molecule_id=mid,error=str(exc)))\n            for col in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[col]=0.\n            parts.append(g)\n        if (index+1)%25==0:print(\'NEURAL SCORED\',index+1,\'molecules\',flush=True)\n    result=pd.concat(parts,ignore_index=True)\n    assert len(result)==len(frame)\n    result.to_parquet(output,index=False)\n    Path(str(output)+\'.errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'NEURAL input failures kept at baseline:\',len(errors),flush=True)\n    print(\'NEURAL FEATURES\',result.shape,checkpoint,flush=True)\n\nif __name__==\'__main__\':score(*sys.argv[1:])\n'
MODULES['submission.py'] = 'import json,hashlib\nfrom pathlib import Path\nimport pandas as pd,numpy as np\nfrom ranking import rrf_order\n\ndef apply(evidence,baseline,output,weight=.25,offset=3.):\n    f=pd.read_parquet(evidence);base=pd.read_csv(baseline,dtype=str,keep_default_na=False)\n    groups={mid:g for mid,g in f.groupby(\'molecule_id\',sort=False)};rows=[];audit=[]\n    assert base.molecule_id.is_unique and set(groups)==set(base.molecule_id)\n    for mid,text in base.itertuples(index=False):\n        g=groups[mid].sort_values(\'baseline_rank\',kind=\'stable\');original=text.split(\';\')\n        assert g.smiles.tolist()==original,mid\n        active=bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n        order=rrf_order(g.neural_score.values,weight,offset) if active else np.arange(len(g))\n        new=g.smiles.iloc[order].tolist()\n        assert len(new)==len(original) and set(new)==set(original),mid\n        assert len(new)<=25 and len(set(new))==len(new)\n        rows.append((mid,\';\'.join(new)))\n        audit.append(dict(molecule_id=mid,processed=active,changed=new!=original,\n            changed_top1=new[0]!=original[0],candidate_count=len(new),weight=weight))\n    result=pd.DataFrame(rows,columns=[\'molecule_id\',\'smiles\'])\n    assert result.molecule_id.tolist()==base.molecule_id.tolist()\n    result.to_csv(output,index=False)\n    out=Path(output).parent;pd.DataFrame(audit).to_csv(out/\'neural_rerank_audit.csv\',index=False)\n    report=dict(method=\'baseline rank + neural rank reciprocal-rank fusion AFTER every baseline rule\',\n        neural_weight=weight,rrf_offset=offset,weight_is_validated=False, validation_scope="Fixed-weight submission experiment; no validation performed",\n        candidate_set_preserved=True,changed_rows=sum(x[\'changed\'] for x in audit),\n        changed_top1=sum(x[\'changed_top1\'] for x in audit),rows=len(rows),\n        frozen_sha256=hashlib.sha256(Path(baseline).read_bytes()).hexdigest(),\n        experimental_sha256=hashlib.sha256(Path(output).read_bytes()).hexdigest())\n    (out/\'neural_submission_report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    return report\n'
MODULES['ranking.py'] = '"""Pure deterministic ranking utilities shared by both notebooks."""\nimport numpy as np\nimport pandas as pd\n\ndef stable_rank(values):\n    values=np.asarray(values,dtype=float)\n    assert np.isfinite(values).all(), \'Nonfinite candidate scores\'\n    order=np.argsort(-values,kind=\'stable\');ranks=np.empty(len(values),int);ranks[order]=np.arange(1,len(values)+1)\n    return ranks\n\ndef rrf_order(neural_scores,weight=0.15,offset=3.0):\n    assert weight>=0 and offset>=0\n    n=len(neural_scores);baseline=np.arange(1,n+1)\n    # Zero weight guarantees exact baseline order even when neural scores are unavailable.\n    if weight==0:return np.arange(n)\n    ranks=stable_rank(neural_scores)\n    return np.argsort(-(1/(offset+baseline)+weight/(offset+ranks)),kind=\'stable\')\n\ndef metrics(frame,score=\'prediction\'):\n    out=[]\n    for mid,g in frame.groupby(\'molecule_id\',sort=False):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\')\n        order=np.argsort(-g[score].to_numpy(),kind=\'stable\')\n        labels=g.label.to_numpy()[order];where=np.flatnonzero(labels)\n        r=int(where[0]+1) if len(where) else None\n        out.append(dict(molecule_id=mid,mrr25=1/r if r and r<=25 else 0.,top1=int(r==1),\n                        candidate_recall=int(bool(len(where))),rank=r))\n    return pd.DataFrame(out)\n\nBASE_FEATURES=[\'baseline_rr\',\'main_rr\',\'prefusion_rr\',\'engine2_rr\',\'pubchem_rr\',\'engine_agreement\',\n    \'library_max\',\'proxy_gl\',\'ice_missing\',\'gl_missing\',\'ice_z\',\'gl_z\',\n    \'neural_cos_z\',\'neural_fp_z\',\'neural_score_z\',\'neural_rr\',\'neural_std\',\n    \'n_spectra_log\',\'n_adducts\',\'neural_top_gap\',\'neural_agrees_top1\',\'engine2_agrees_top1\',\n    \'neural_delta_rank\',\'neural_x_library\',\'neural_x_agreement\',\'neural_x_sim_missing\',\n    \'main_missing\',\'engine2_missing\',\'pubchem_missing\']\n\ndef make_features(frame):\n    f=frame.copy()\n    for src in [\'baseline\',\'main\',\'prefusion\',\'engine2\',\'pubchem\']:\n        r=pd.to_numeric(f[src+\'_rank\'],errors=\'coerce\');f[src+\'_rr\']=np.where(r>0,1/(3+r),0)\n        if src in [\'main\',\'engine2\',\'pubchem\']:f[src+\'_missing\']=(~(r>0)).astype(float)\n    for src in [\'ice\',\'gl\',\'neural_cos\',\'neural_fp\',\'neural_score\']:\n        v=pd.to_numeric(f[src],errors=\'coerce\');v=v.where(np.isfinite(v))\n        if src in [\'ice\',\'gl\']:f[src+\'_missing\']=v.isna().astype(float)\n        grp=v.groupby(f.molecule_id,sort=False)\n        f[src+\'_z\']=((v-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    f[\'neural_rank\']=0.;f[\'neural_top_gap\']=0.;f[\'neural_agrees_top1\']=0.;f[\'engine2_agrees_top1\']=0.\n    for _,g in f.groupby(\'molecule_id\',sort=False):\n        idx=g.index;rank=stable_rank(g.neural_score.values);f.loc[idx,\'neural_rank\']=rank\n        vals=np.sort(g.neural_score_z.values)[::-1]\n        f.loc[idx,\'neural_top_gap\']=float(vals[0]-vals[1]) if len(vals)>1 else 0.\n        f.loc[idx,\'neural_agrees_top1\']=float(g.iloc[np.argmin(rank)].baseline_rank==1)\n        f.loc[idx,\'engine2_agrees_top1\']=float(((g.baseline_rank==1)&(g.engine2_rank==1)).any())\n    f[\'neural_rr\']=1/(3+f.neural_rank)\n    f[\'engine_agreement\']=((f.main_rank>0).astype(int)+(f.engine2_rank>0).astype(int)+(f.pubchem_rank>0).astype(int))\n    f[\'n_spectra_log\']=np.log1p(f.n_spectra)\n    f[\'neural_delta_rank\']=(f.baseline_rank-f.neural_rank)/25\n    f[\'neural_x_library\']=f.neural_score_z*f.library_max\n    f[\'neural_x_agreement\']=f.neural_score_z*f.engine_agreement\n    f[\'neural_x_sim_missing\']=f.neural_score_z*f.ice_missing*f.gl_missing\n    X=f[BASE_FEATURES].astype(float).replace([np.inf,-np.inf],np.nan).fillna(0.)\n    assert np.isfinite(X.values).all()\n    return X\n\ndef residual_scores(frame,probability,beta):\n    p=np.clip(np.asarray(probability,float),1e-5,1-1e-5)\n    raw=pd.Series(np.log(p/(1-p)),index=frame.index)\n    grp=raw.groupby(frame.molecule_id,sort=False)\n    z=((raw-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    correction=float(beta)*z.values\n    if \'neural_ok\' in frame:correction=np.where(frame.neural_ok.values,correction,0.)\n    return -np.log(frame.baseline_rank.values.astype(float))+correction\n'
MODULES['bundle.py'] = "import hashlib,json,zipfile\nfrom pathlib import Path\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for b in iter(lambda:f.read(8<<20),b''):h.update(b)\n    return h.hexdigest()\n\ndef find_bundle(input_root,output_root,pins):\n    roots=[]\n    for path in Path(input_root).rglob('final_model.pt'):\n        if (path.parent/'config.json').is_file():roots.append(path.parent)\n    if not roots:\n        archives=list(Path(input_root).rglob('casmi_spectrum_model_v1.zip'))\n        assert len(archives)==1, 'Attach the supplied casmi_spectrum_model_v1.zip as a Kaggle dataset (zipped or extracted)'\n        dest=Path(output_root)/'uploaded_model';dest.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for item in z.infolist():\n                target=(dest/item.filename).resolve()\n                assert target.is_relative_to(dest.resolve()),'Unsafe ZIP member'\n            z.extractall(dest)\n        roots=[p.parent for p in dest.rglob('final_model.pt')]\n    matching=[r for r in roots if all((r/n).is_file() and sha(r/n)==h for n,h in pins.items())]\n    assert len(matching)==1, f'Expected exactly one matching model bundle, found {len(matching)}. Use the attached v1 artifacts.'\n    return matching[0]\n"
MODULES['consensus.py'] = '"""Leader-safe consensus re-ranking for CASMI26 answer lists.\n\nRuns after the frozen 0.465 pipeline and after the neural scorer. It never reruns a\nmodel: it reorders the candidates of each molecule and, only when asked, lets a few\ncandidates that the pipeline left outside its list compete for the last places.\n\nOrder rule\n    rank 1   = leader of   1/(k + b) + w_top / (k + n)\n    ranks 2+ = the others, ordered by   1/(k + b) + w_mid / (k + n)\nwith b the baseline rank, n the neural rank among the same candidates and k the\noffset. With w_top == w_mid this is exactly the fixed-weight reciprocal-rank fusion\nof the public 0.466 notebook (``ranking.rrf_order``), which build_variants re-checks.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom ranking import rrf_order, stable_rank\n\nPRESETS = {\n    \'V0\': dict(w_top=0.00, w_mid=0.00, rescue=0, label=\'Baseline 0.465 - no neural term\'),\n    \'V1\': dict(w_top=0.25, w_mid=0.25, rescue=0, label=\'Neural RRF 0.25 - public 0.466 recipe\'),\n    \'V2\': dict(w_top=0.25, w_mid=0.50, rescue=0, label=\'Leader-safe consensus 0.25 / 0.50\'),\n    \'V3\': dict(w_top=0.25, w_mid=0.50, rescue=3, label=\'Leader-safe consensus + 3-slot tail rescue\'),\n    \'V4\': dict(w_top=0.25, w_mid=1.00, rescue=0, label=\'Bold consensus 0.25 / 1.00\'),\n}\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef leader_safe_order(neural_scores, w_top, w_mid, offset=3.0):\n    """Indices of the new order (baseline order = 0..n-1)."""\n    s = np.asarray(neural_scores, dtype=float)\n    if len(s) == 0:\n        return np.zeros(0, dtype=int)\n    first = np.asarray(rrf_order(s, w_top, offset), dtype=int)\n    if w_mid == w_top:\n        return first\n    lead = int(first[0])\n    rest = [int(i) for i in rrf_order(s, w_mid, offset) if int(i) != lead]\n    return np.asarray([lead] + rest, dtype=int)\n\n\ndef rescue_tail(smiles, scores, outsiders, slots, limit=25):\n    """Let outside candidates compete for the last `slots` places by neural score.\n\n    smiles, scores: the reordered list and its neural scores.\n    outsiders: [(smiles, neural_score)] whose metric identities differ from every\n    listed candidate and from each other (guaranteed by outsiders.py).\n    Free places of a list shorter than `limit` are offered as well.\n    Returns (new_list, number_of_outsiders_admitted).\n    """\n    smiles = list(smiles)\n    scores = [float(v) for v in scores]\n    if slots <= 0 or not outsiders or not smiles:\n        return smiles, 0\n    k = min(int(slots), len(smiles))\n    free = max(0, int(limit) - len(smiles))\n    head = smiles[:len(smiles) - k]\n    pool = [(s, v, 0) for s, v in zip(smiles[len(smiles) - k:], scores[len(scores) - k:])]\n    pool += [(str(s), float(v), 1) for s, v in outsiders if np.isfinite(float(v)) and str(s) not in smiles]\n    pool.sort(key=lambda t: (-t[1], t[2]))  # stable; on ties a listed candidate keeps its place\n    chosen = pool[:k + free]\n    out = (head + [s for s, _, _ in chosen])[:limit]\n    return out, int(sum(t[2] for t in chosen))\n\n\ndef read_outsiders(path):\n    out = {}\n    if not path or not Path(path).is_file():\n        return out\n    frame = pd.read_parquet(path)\n    if frame.empty or \'neural_ok\' not in frame.columns:\n        return out\n    score = pd.to_numeric(frame.neural_score, errors=\'coerce\')\n    frame = frame[frame.neural_ok.astype(bool) & np.isfinite(score)]\n    for mid, g in frame.groupby(\'molecule_id\', sort=False):\n        g = g.sort_values(\'baseline_rank\', kind=\'stable\')\n        out[str(mid)] = list(zip(g.smiles.astype(str), g.neural_score.astype(float)))\n    return out\n\n\ndef build_variants(evidence, baseline, outdir, presets=None, offset=3.0, outsiders=None):\n    """Write submission_<name>.csv for every preset; return (report, moves)."""\n    presets = dict(PRESETS if presets is None else presets)\n    frame = pd.read_parquet(evidence)\n    base = pd.read_csv(baseline, dtype=str, keep_default_na=False)\n    assert list(base.columns) == [\'molecule_id\', \'smiles\'] and base.molecule_id.is_unique\n    groups = {str(m): g.sort_values(\'baseline_rank\', kind=\'stable\') for m, g in frame.groupby(\'molecule_id\', sort=False)}\n    assert set(groups) == set(base.molecule_id), \'evidence and baseline cover different molecules\'\n    outs = read_outsiders(outsiders)\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    report, moves = {}, []\n    for name, cfg in presets.items():\n        rows = []\n        stats = dict(active_rows=0, changed_rows=0, changed_top1=0, rescued=0)\n        for mid, text in base.itertuples(index=False):\n            original = text.split(\';\')\n            g = groups[mid]\n            assert g.smiles.tolist() == original, f\'evidence order differs from the baseline: {mid}\'\n            active = bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n            if active:\n                order = leader_safe_order(g.neural_score.values, cfg[\'w_top\'], cfg[\'w_mid\'], offset)\n            else:\n                order = np.arange(len(g))\n            new = g.smiles.iloc[order].tolist()\n            new_scores = g.neural_score.astype(float).values[order].tolist()\n            rescued = 0\n            if active and int(cfg.get(\'rescue\', 0)) > 0 and outs.get(mid):\n                new, rescued = rescue_tail(new, new_scores, outs[mid], int(cfg[\'rescue\']))\n            assert 1 <= len(new) <= 25 and len(set(new)) == len(new), mid\n            if not rescued:\n                assert sorted(new) == sorted(original), f\'candidate set changed: {mid}\'\n            stats[\'active_rows\'] += int(active)\n            stats[\'changed_rows\'] += int(new != original)\n            stats[\'changed_top1\'] += int(new[0] != original[0])\n            stats[\'rescued\'] += int(rescued)\n            rows.append((mid, \';\'.join(new)))\n            if active:\n                b_rank = {s: i + 1 for i, s in enumerate(original)}\n                n_rank = dict(zip(g.smiles, stable_rank(g.neural_score.values)))\n                for pos, s in enumerate(new, 1):\n                    moves.append((name, mid, s, pos, b_rank.get(s, 0), int(n_rank.get(s, 0))))\n        out = pd.DataFrame(rows, columns=[\'molecule_id\', \'smiles\'])\n        assert out.molecule_id.tolist() == base.molecule_id.tolist()\n        path = outdir / f\'submission_{name}.csv\'\n        out.to_csv(path, index=False)\n        report[name] = dict(cfg, offset=offset, file=path.name, sha256=sha256(path), **stats)\n    moves = pd.DataFrame(moves, columns=[\'variant\', \'molecule_id\', \'smiles\', \'final_rank\', \'baseline_rank\', \'neural_rank\'])\n    (outdir / \'consensus_report.json\').write_text(json.dumps(report, indent=2))\n    return report, moves\n\n\ndef validate(path, sample_ids, baseline_path=None, allow_new=False, identity_of=None):\n    """Raise unless `path` is a well-formed answer file for the expected molecules."""\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    assert list(frame.columns) == [\'molecule_id\', \'smiles\'], \'columns\'\n    assert frame.molecule_id.tolist() == list(sample_ids), \'molecule order\'\n    lists = frame.smiles.str.split(\';\')\n    assert lists.map(len).between(1, 25).all(), \'list length\'\n    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), \'empty or repeated SMILES\'\n    if baseline_path is not None:\n        base = pd.read_csv(baseline_path, dtype=str, keep_default_na=False)\n        assert base.molecule_id.tolist() == frame.molecule_id.tolist(), \'baseline order\'\n        for before, after in zip(base.smiles, frame.smiles):\n            b, a = before.split(\';\'), after.split(\';\')\n            if allow_new:\n                assert len(a) >= min(len(b), 25), \'list got shorter\'\n            else:\n                assert sorted(b) == sorted(a), \'candidate set changed\'\n    if identity_of is not None:\n        for answer in frame.smiles:\n            keys = [identity_of.get(s) for s in answer.split(\';\')]\n            keys = [k for k in keys if k]\n            assert len(keys) == len(set(keys)), \'duplicate metric identity\'\n    return frame\n'
MODULES['outsiders.py'] = '"""Collect candidates that the frozen pipeline considered but left outside its final list.\n\nReads the pipeline\'s own record of its candidate lists (e004_candidate_inputs.json),\nwalks the fusion overflow, Engine 2, the main-engine tail and the PubChem channel in\nround-robin rank order, and keeps structures whose official metric identity (the frozen\nnotebook\'s own metric_identity, executed from its source) is new for the molecule.\nOutput rows follow the evidence schema the neural scorer reads; baseline_rank 26, 27, ...\nonly fixes their order.\n\nusage: python outsiders.py <frozen.ipynb> <root> <evidence.parquet> <out.parquet> [per_molecule] [max_checks]\n"""\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nCOLUMNS = [\'molecule_id\', \'smiles\', \'metric_key\', \'source\', \'source_rank\', \'baseline_rank\', \'processed\']\n\n\ndef load_identity(frozen):\n    cells = json.loads(Path(frozen).read_text())[\'cells\']\n    hits = [\'\'.join(c[\'source\']) for c in cells\n            if c[\'cell_type\'] == \'code\' and \'def metric_identity\' in \'\'.join(c[\'source\'])]\n    assert len(hits) == 1, \'metric identity cell not found in the frozen source\'\n    scope = {\'__name__\': \'frozen_metric_identity\'}\n    exec(compile(hits[0], \'frozen_metric_identity\', \'exec\'), scope)\n    return scope[\'metric_identity\']\n\n\ndef source_lists(payload, mid):\n    eng = payload.get(\'engine2\', {}).get(mid) or {}\n    pc = payload.get(\'pubchem\', {}).get(mid) or {}\n    return [(\'fusion_overflow\', list(payload.get(\'fusion_overflow\', {}).get(mid) or [])),\n            (\'engine2\', list(eng.get(\'smiles\') or [])),\n            (\'main_tail\', list(payload.get(\'base_tail\', {}).get(mid) or [])),\n            (\'pubchem\', list(pc.get(\'pc\') or []))]\n\n\ndef collect(payload, evidence, identity, per_molecule=12, max_checks=30):\n    rows = []\n    active = evidence[evidence.processed.astype(bool)]\n    for mid, g in active.groupby(\'molecule_id\', sort=False):\n        mid = str(mid)\n        if mid not in payload.get(\'prefusion\', {}):\n            continue\n        listed = g.sort_values(\'baseline_rank\', kind=\'stable\').smiles.astype(str).tolist()\n        taken = {k for k in g.metric_key.tolist() if isinstance(k, str) and k}\n        seen = set(listed)\n        lists = source_lists(payload, mid)\n        depth = max((len(v) for _, v in lists), default=0)\n        picked = checks = 0\n        for r in range(depth):\n            if picked >= per_molecule or checks >= max_checks:\n                break\n            for name, values in lists:\n                if r >= len(values) or picked >= per_molecule or checks >= max_checks:\n                    continue\n                s = str(values[r]).strip()\n                if not s or s == \'CCO\' or s in seen:\n                    continue\n                seen.add(s)\n                checks += 1\n                key = identity(s)\n                if key is None or key[0] in taken:\n                    continue\n                taken.add(key[0])\n                picked += 1\n                rows.append((mid, s, key[0], name, r + 1, 25 + picked, True))\n    return pd.DataFrame(rows, columns=COLUMNS)\n\n\nif __name__ == \'__main__\':\n    frozen, root, evidence_path, out = sys.argv[1:5]\n    per = int(sys.argv[5]) if len(sys.argv) > 5 else 12\n    checks = int(sys.argv[6]) if len(sys.argv) > 6 else 30\n    t0 = time.time()\n    identity = load_identity(frozen)\n    payload = json.loads((Path(root) / \'e004_candidate_inputs.json\').read_text())\n    evidence = pd.read_parquet(evidence_path)\n    frame = collect(payload, evidence, identity, per, checks)\n    frame.to_parquet(out, index=False)\n    by_source = frame.source.value_counts().to_dict() if len(frame) else {}\n    print(\'OUTSIDERS\', len(frame), \'candidates for\', frame.molecule_id.nunique(), \'molecules\',\n          json.dumps(by_source), f\'{time.time() - t0:.0f}s\', flush=True)\n'
MODULE_SHA256 = {
 "runner.py": "86b74f3b4cd5767a2cbe1d291735c292b66b4fbf286a705c6cb777a1865883db",
 "export_evidence.py": "d4ec36a9f6216d2607fff4f481bc465dbc8fbc616ec4b24f12932e4c5e546e19",
 "neural_scores.py": "cb9e4d6732992e3310e951533fd0c644d3b515134ad03768c35e5a86551267d2",
 "submission.py": "d0027a893b1bfefe8066ede4f625c584fe0d4fa16bfe4bee32265ef34a7f007c",
 "ranking.py": "6f909cd7e41e88387c23109b93dd590885d2283509804279a58c624a758987d7",
 "bundle.py": "8595ef16ead9c9e6ac34a3398bff4b748f5668cdc517c396be18100a5b745cea",
 "consensus.py": "9e67ab49a852404587e62ca84e0fc551d70ce68ec8f7a47739aada6ca08d3989",
 "outsiders.py": "4dcf6c2b9c9b7be3ac8f7a616683495430fcb4e0c25b449684d132daaa8e7b5c"
}
MODEL_PINS = {
 "features.py": "a97ad9b3ab3711517864d56e497e1672cb29bb05683c72b52db9f9b0a17b1455",
 "model.py": "2547418857386d1c2513fd1ab6c70961e6a96e73d9d495bbfe3052197e69dc39",
 "config.json": "563fb2a2a81ca48ea094f9ab78c157a48a6e9a5503e85e6ce8f0fb3615c1cc03",
 "final_model.pt": "44436b2daec64373e66f9120dd2389c24fc83dd9430ebeeac18e3cb49dcb9642"
}
FROZEN_SHA256 = 'd1e757c2b71b46e6e8a5777f14302a4737d0fdbcd56d8365093c039b93c5fd88'
FROZEN_B64 = '...'
print(f'{len(MODULES)} modules and the pinned pipeline source ready ({len(FROZEN_B64) * 3 // 4 / 1e6:.2f} MB)')

In [ ]:
# ── Stage machinery: process runner, compact trace, and the four stage functions ───────────────
import contextlib, importlib

STAGES = []           # (stage, seconds, status) for the run summary
STAGE_MARKS = {}      # stage -> [(frozen cell, START/DONE, seconds since stage start)]
FROZEN_STAGE = {
    2: 'Official RDKit 2026.03.3', 6: 'Competition files', 9: 'Asset inventory', 12: 'Documentation prior table',
    14: 'Prior benchmark table', 16: 'Natural-product panel', 18: 'Natural-product knowledge table',
    26: 'Datasets, priors and wheels', 27: 'Determinism patch', 28: 'Preflight', 30: 'Commit or rerun switch',
    32: 'Engine 2 · two rankers', 34: 'PubChem channel', 36: 'Engine 1 · library, pool, spectrum cache',
    37: 'Engine 1 · models and rankers', 39: 'Engine 1 · candidate lists', 41: 'GLACIER + ICEBERG forward models',
    43: 'Gated PubChem merge', 45: 'Two-engine fusion', 48: 'Metric audit and safe tail', 49: 'Metric-identity fusion',
    50: 'Postflight record', 51: 'Fragment score gap', 52: 'Window-40 overlay', 53: 'Engine 2 rank-2 placement',
    54: 'Execution receipt', 56: 'Example answer table'}
CELL_MARK = re.compile(r'^PIPELINE CELL (\d+) (START|DONE)')
KEY_LINE = re.compile(r'FAILED|ERROR|Traceback|merge stats|RANK2_OVERLAY|EXPORTED FROZEN|NEURAL FEATURES|'
                      r'NEURAL input failures|OUTSIDERS|engine lists \d+ |pubchem lists \d+|errors \d+ \|')
BASELINE = ROOT / 'submission_baseline0465.csv'
VARIANTS_DIR = ROOT / 'variants'
RESCUE_SLOTS = {'V3': 3, 'CUSTOM': int(CUSTOM.get('rescue', 0))}.get(VARIANT, 0)
PIPELINE_DONE = NEURAL_OK = OUTSIDERS_OK = CONSENSUS_OK = False
REPORT, MOVES, SUBMITTED, consensus = {}, pd.DataFrame(), None, None


def run_stage(args, log, name, quiet=None):
    """Run one stage in its own process; keep the full log on disk and print a compact trace."""
    quiet = QUIET_LOGS if quiet is None else quiet
    env = {**os.environ, 'PYTHONHASHSEED': '0', 'PYTHONUNBUFFERED': '1'}
    env.pop('INC_PANEL', None)
    t0, marks, tail = time.time(), [], collections.deque(maxlen=150)
    print(f'▶ {name}', flush=True)
    with open(log, 'w') as handle:
        proc = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            handle.write(line); handle.flush()
            t = time.time() - t0
            tail.append(line)
            m = CELL_MARK.match(line)
            if m:
                cell, kind = int(m.group(1)), m.group(2)
                marks.append((cell, kind, t))
                if quiet and kind == 'DONE' and cell in FROZEN_STAGE:
                    start = next((s for c, k, s in reversed(marks) if c == cell and k == 'START'), t)
                    if t - start >= 1:
                        print(f'  ✓ {t / 60:6.1f} min   {FROZEN_STAGE[cell]:<40s} {t - start:8,.0f} s', flush=True)
                continue
            if not quiet:
                if len(line) < 5000:
                    print(line, end='', flush=True)
            elif KEY_LINE.search(line) and len(line) < 800:
                print('      ' + line.strip()[:180], flush=True)
        status = proc.wait()
    seconds = time.time() - t0
    STAGE_MARKS[name] = marks
    STAGES.append((name, seconds, 'ok' if status == 0 else f'exit {status}'))
    if status:
        print(''.join(tail))
        raise RuntimeError(f'{name} failed with exit code {status}; full log: {log}')
    print(f'  done in {minutes(seconds)}', flush=True)
    return seconds


def materialise_sources():
    """Stage 0: write and verify the pinned sources, find the neural bundle, install the official RDKit."""
    global FROZEN
    t0 = time.time()
    for name, text in MODULES.items():
        (CODE / name).write_text(text)
        assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == MODULE_SHA256[name], name
    FROZEN = CODE / 'frozen_0465.ipynb'
    FROZEN.write_bytes(base64.b64decode(FROZEN_B64))
    assert hashlib.sha256(FROZEN.read_bytes()).hexdigest() == FROZEN_SHA256, 'Pinned pipeline source differs'
    assert sys.version_info[:2] in ((3, 12), (3, 13)), 'Python 3.12 or 3.13 is required'
    if str(CODE) not in sys.path:
        sys.path.insert(0, str(CODE))
    from bundle import find_bundle
    bundle = find_bundle(str(INPUT), ROOT, MODEL_PINS)
    (CODE / 'bootstrap.py').write_text(''.join(json.loads(FROZEN.read_text())['cells'][2]['source']))
    run_stage([sys.executable, CODE / 'bootstrap.py'], ROOT / 'bootstrap.log', 'Official RDKit wheel', quiet=False)
    STAGES[-1] = ('Sources and RDKit wheel', time.time() - t0, 'ok')
    tiles([('Pipeline source', FROZEN_SHA256[:12] + '…', 'SHA-256 verified'),
           ('Helper modules', f'{len(MODULES)}', '6 public + 2 new, SHA-256 verified'),
           ('Neural bundle', Path(bundle).name, '4 files, SHA-256 verified'),
           ('Stage 0', minutes(time.time() - t0), 'including the wheel install')])
    return bundle


def run_public_pipeline():
    """Stage 1: every cell of the pinned pipeline in one process, then the export of its final lists."""
    if not RUN_PIPELINE:
        note('Fast commit.', 'The pipeline is skipped in this commit run; the scoring rerun executes every stage.', BLUE)
        return False
    assert not (ROOT / 'work/spec_meta.parquet').exists() and not (ROOT / 'submission.csv').exists(), \
        'Start from a fresh session (Run All after a restart)'
    seconds = run_stage([sys.executable, CODE / 'runner.py', FROZEN, FROZEN_SHA256, ROOT],
                        ROOT / 'submission_pipeline.log', 'Public 0.465 pipeline')
    receipt = json.loads((ROOT / 'execution_receipt.json').read_text())
    full = receipt['is_full_run']
    tiles([('Molecules ranked', f"{receipt['processed_molecules']:,}", 'hidden test' if full else 'visible smoke subset'),
           ('Answer rows', f'{len(SAMPLE_IDS):,}', 'all ranked' if full else 'placeholders elsewhere'),
           ('Pipeline time', minutes(seconds), 'one process, every cell'),
           ('Baseline file', BASELINE.name, 'kept for reference')])
    return True


def score_final_lists():
    """Stage 2a: the public scorer on exactly the exported candidates."""
    try:
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'frozen_candidate_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'neural_candidate_features.parquet', 'cuda'],
                  ROOT / 'submission_neural.log', 'Neural scorer · final lists')
        return True
    except Exception as exc:
        note('Neural scorer failed.', f'The 0.465 answers will be submitted. {type(exc).__name__}: {str(exc)[:300]}', RED)
        return False


def score_outside_candidates(needed):
    """Stage 2b (optional): collect and score candidates ranked just outside the final lists."""
    elapsed_h = (time.time() - NB_T0) / 3600
    if not needed:
        return False
    if elapsed_h >= TIME_GUARD_HOURS:
        note('Tail rescue skipped.', f'The session has run {elapsed_h:.1f} h (guard {TIME_GUARD_HOURS} h).', YELLOW)
        return False
    try:
        run_stage([sys.executable, CODE / 'outsiders.py', FROZEN, ROOT, ROOT / 'frozen_candidate_evidence.parquet',
                   ROOT / 'outsider_evidence.parquet', OUTSIDERS_PER_MOLECULE],
                  ROOT / 'outsiders.log', 'Outside candidates · collect')
        if not len(pd.read_parquet(ROOT / 'outsider_evidence.parquet')):
            return False
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'outsider_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'outsider_neural_features.parquet', 'cuda'],
                  ROOT / 'outsiders_neural.log', 'Neural scorer · outside candidates')
        return True
    except Exception as exc:
        note('Tail rescue disabled.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
        return False


def show_neural_summary():
    if not NEURAL_OK:
        return
    nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet')
    act = nf[nf.processed.astype(bool)]
    n_out = len(pd.read_parquet(ROOT / 'outsider_neural_features.parquet')) if OUTSIDERS_OK else 0
    tiles([('Candidates scored', f'{len(act):,}', f'{act.molecule_id.nunique():,} molecules'),
           ('Scored molecules', f'{int(act.groupby("molecule_id").neural_ok.all().sum()):,}', 'all spectra usable'),
           ('Spectra per molecule', f'{act.groupby("molecule_id").n_spectra.first().median():.0f}', 'median used by the scorer'),
           ('Outside candidates', f'{n_out:,}', 'scored for the tail rescue' if n_out else 'not needed for this variant')])


def build_consensus():
    """Stage 3a: the public 0.466 file, its exact rebuild, then every variant."""
    global consensus
    try:
        consensus = importlib.import_module('consensus')
        from submission import apply as public_apply
        evidence = ROOT / 'neural_candidate_features.parquet'
        with contextlib.redirect_stdout(io.StringIO()):      # the public helper prints its own report
            public_apply(evidence, BASELINE, ROOT / 'submission_public0466.csv', weight=0.25, offset=3.0)
        consensus.build_variants(evidence, BASELINE, ROOT / 'variants_check', {'V1': consensus.PRESETS['V1']}, 3.0)
        mine = pd.read_csv(ROOT / 'variants_check/submission_V1.csv', dtype=str, keep_default_na=False)
        public = pd.read_csv(ROOT / 'submission_public0466.csv', dtype=str, keep_default_na=False)
        assert mine.equals(public), 'V1 differs from the public 0.466 file'
        presets = dict(consensus.PRESETS)
        if VARIANT == 'CUSTOM':
            presets['CUSTOM'] = dict(w_top=float(CUSTOM['w_top']), w_mid=float(CUSTOM['w_mid']),
                                     rescue=int(CUSTOM.get('rescue', 0)), label='Custom weights')
        report, moves = consensus.build_variants(
            evidence, BASELINE, VARIANTS_DIR, presets, RRF_OFFSET,
            ROOT / 'outsider_neural_features.parquet' if OUTSIDERS_OK else None)
        print('V1 rebuilt exactly: the new code reproduces the public 0.466 file row for row.')
        return True, report, moves
    except Exception as exc:
        note('Consensus layer disabled.', f'{type(exc).__name__}: {str(exc)[:400]}', RED)
        return False, {}, pd.DataFrame()


def identity_map():
    keys = {}
    for path in (ROOT / 'neural_candidate_features.parquet', ROOT / 'outsider_neural_features.parquet'):
        if path.is_file():
            f = pd.read_parquet(path, columns=['smiles', 'metric_key'])
            keys.update({s: k for s, k in zip(f.smiles, f.metric_key) if isinstance(k, str)})
    return keys


def well_formed(path, same_candidates=True):
    """Minimal checks that do not depend on the new modules."""
    f = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, 'rows'
    lists = f.smiles.str.split(';')
    assert lists.map(len).between(1, 25).all(), 'list length'
    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), 'SMILES'
    if same_candidates and BASELINE.is_file():
        b = pd.read_csv(BASELINE, dtype=str, keep_default_na=False)
        assert all(sorted(x.split(';')) == sorted(y.split(';')) for x, y in zip(b.smiles, f.smiles)), 'candidates'


def choose_and_submit():
    """Stage 3b: chosen variant -> V1 -> V0 -> public 0.466 file -> baseline file (-> sample in a fast commit)."""
    t0 = time.time()
    chain = []
    if CONSENSUS_OK:
        chain += [(n, VARIANTS_DIR / f'submission_{n}.csv') for n in dict.fromkeys([VARIANT, 'V1', 'V0'])]
    chain += [('public 0.466 file', ROOT / 'submission_public0466.csv'), ('baseline file', BASELINE)]
    chosen = None
    for name, path in chain:
        if not path.is_file():
            continue
        try:
            rescue = name in REPORT and REPORT[name].get('rescued', 0) > 0
            if rescue:
                consensus.validate(path, SAMPLE_IDS, BASELINE, allow_new=True, identity_of=identity_map())
            well_formed(path, same_candidates=not rescue)
            chosen = (name, path)
            break
        except Exception as exc:
            note(f'{name} rejected.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
    if chosen is None:
        assert not IS_RERUN, 'No valid answer file in the scoring rerun'
        chosen = ('sample (fast commit)', COMP / 'sample_submission.csv')
    shutil.copyfile(chosen[1], ROOT / 'submission.csv')
    STAGES.append(('Consensus and checks', time.time() - t0, 'ok'))
    return chosen[0]


def show_variant_board():
    if CONSENSUS_OK:
        board = pd.DataFrame([{
            'Variant': k, 'Rule': v['label'], 'Leader weight': f"{v['w_top']:.2f}",
            'Ranks 2–25 weight': f"{v['w_mid']:.2f}", 'Tail rescue': v['rescue'] or '–',
            'Rows re-ordered': f"{v['changed_rows']:,}", 'Rank 1 changed': f"{v['changed_top1']:,}",
            'Rescued': f"{v['rescued']:,}"} for k, v in REPORT.items()])
        board.index = board.Variant
        table_html(board, highlight=SUBMITTED, numeric=board.columns[2:])
    sub = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    tiles([('Submitted', SUBMITTED, 'copied to submission.csv'),
           ('Rows', f'{len(sub):,}', 'one per molecule, sample order'),
           ('Candidates per row', f'{sub.smiles.str.count(";").add(1).median():.0f}', 'median, at most 25'),
           ('SHA-256', hashlib.sha256((ROOT / 'submission.csv').read_bytes()).hexdigest()[:12] + '…', 'of submission.csv')])

In [ ]:
from pathlib import Path
_physical = '"""Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\'s <=2-bond-cleavage fragments explain.\n\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\nremember the fewest cleavages that give each mass, then match every peak against\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\nA peak\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\nand a spectrum\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\n\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\n\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\'s periodic table\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\nE = 0.00054857990\nPROTON = H - E\nH2O = 2 * H + AMU[\'O\']\nNH4 = AMU[\'N\'] + 4 * H\nFORMATE = AMU[\'C\'] + 2 * H + 2 * AMU[\'O\']\n\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\nCARRIERS = {\n    \'[M+H]+\': (PROTON,), \'[M-H2O+H]+\': (PROTON,), \'[M-2H2O+H]+\': (PROTON,),\n    \'[M+NH4]+\': (PROTON, NH4 - E), \'[M+Na]+\': (PROTON, AMU[\'Na\'] - E), \'[M+K]+\': (PROTON, AMU[\'K\'] - E),\n    \'[M-H]-\': (-PROTON,), \'[M-H2O-H]-\': (-PROTON,), \'[M+CH2O2-H]-\': (-PROTON, FORMATE - PROTON),\n    \'[M+Cl]-\': (-PROTON, AMU[\'Cl\'] + E),\n}\n\nDEFAULT_CFG = dict(\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\n    intensity=\'linear\',          # \'linear\' | \'sqrt\' | a float power applied to `it`\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\n    tol=0.005,                   # absolute m/z tolerance, Da\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\n    max_breaks=2,                # bond cleavages per fragment (1..3)\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\n)\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\'sqrt\', tol=0.01, h_shifts=tuple(range(-2, 3)))\n\n\ndef resolve_cfg(cfg=None):\n    """DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently)."""\n    out = dict(DEFAULT_CFG)\n    for k, v in (cfg or {}).items():\n        if k not in out:\n            raise KeyError(f\'frag_rescore: unknown cfg key {k!r}\')\n        out[k] = v\n    out[\'h_shifts\'] = tuple(int(d) for d in out[\'h_shifts\'])\n    if not 1 <= int(out[\'max_breaks\']) <= 3:\n        raise ValueError(\'frag_rescore: max_breaks must be 1, 2 or 3\')\n    return out\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\n# ---------------------------------------------------------------------------------------------------------------\ndef _find(parent, a):\n    while parent[a] != a:\n        parent[a] = parent[parent[a]]\n        a = parent[a]\n    return a\n\n\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\n    """Union the kept bonds; if the graph splits, append each piece\'s mass with cleavage count k."""\n    for a in range(n):\n        parent[a] = a\n        acc[a] = 0.0\n    ncomp = n\n    for e in range(ea.shape[0]):\n        if removed[e]:\n            continue\n        ra = _find(parent, ea[e])\n        rb = _find(parent, eb[e])\n        if ra != rb:\n            parent[ra] = rb\n            ncomp -= 1\n    if ncomp < 2:\n        return cnt\n    for a in range(n):\n        r = _find(parent, a)\n        acc[r] += w[a]\n    for a in range(n):\n        if parent[a] == a:\n            out_m[cnt] = acc[a]\n            out_k[cnt] = k\n            cnt += 1\n    return cnt\n\n\ndef _enumerate(n, w, ea, eb, max_breaks, base_components):\n    """All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces."""\n    nb = ea.shape[0]\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\n    out_m = np.empty(cap, np.float64)\n    out_k = np.empty(cap, np.int8)\n    parent = np.empty(n, np.int64)\n    acc = np.empty(n, np.float64)\n    removed = np.zeros(nb, np.bool_)\n    cnt = 0\n    for i in range(nb):\n        removed[i] = True\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\n        if max_breaks >= 2:\n            for j in range(i + 1, nb):\n                removed[j] = True\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 2)\n                if max_breaks >= 3:\n                    for l in range(j + 1, nb):\n                        removed[l] = True\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\n                        removed[l] = False\n                removed[j] = False\n        removed[i] = False\n    return out_m[:cnt], out_k[:cnt]\n\n\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\n    from numba import njit\n    _find = njit(cache=True, nogil=True)(_find)\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\n    HAVE_NUMBA = True\nexcept Exception:  # pragma: no cover - fallback path\n    HAVE_NUMBA = False\n\n\ndef mol_graph(smi):\n    """(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    try:\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\n    except Exception:\n        m = None\n    if m is None or m.GetNumAtoms() == 0:\n        return None\n    pt = Chem.GetPeriodicTable()\n    w = np.empty(m.GetNumAtoms(), np.float64)\n    for a in m.GetAtoms():\n        s = a.GetSymbol()\n        am = AMU.get(s)\n        if am is None:\n            am = pt.GetMostCommonIsotopeMass(s)\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\n    return w, ea, eb, len(Chem.GetMolFrags(m))\n\n\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40):\n    """Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\n\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\n    SMILES does not parse.\n    """\n    g = mol_graph(smi)\n    if g is None:\n        return None\n    w, ea, eb, npieces = g\n    nb = len(ea)\n    mb = int(max_breaks)\n    if mb >= 3 and nb > max_bonds_3:\n        mb = 2\n    if mb >= 2 and nb > max_bonds_2:\n        mb = 1\n    levels = [np.array([w.sum()])]\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smi)\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\n    if nb == 0:\n        return levels + [np.zeros(0) for _ in range(int(max_breaks))]\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces)\n    fm = np.concatenate([levels[0], fm])\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\n    key = np.round(fm, 5)\n    order = np.lexsort((fk, key))                  # by mass, then fewest cleavages first\n    key, fm, fk = key[order], fm[order], fk[order]\n    first = np.ones(len(key), bool)\n    first[1:] = key[1:] != key[:-1]\n    fm, fk = fm[first], fk[first]\n    return [np.sort(fm[fk == k]) for k in range(int(max_breaks) + 1)]\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# scoring\n# ---------------------------------------------------------------------------------------------------------------\ndef _carriers(adduct, mode):\n    c = CARRIERS.get(adduct)\n    if c is not None:\n        return c\n    if isinstance(adduct, str) and adduct.endswith(\'-\'):\n        return (-PROTON,)\n    if isinstance(adduct, str) and adduct.endswith(\'+\'):\n        return (PROTON,)\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\n\n\ndef _prep_spectrum(s, c):\n    """Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair."""\n    mz = np.asarray(s.get(\'mz\', ()), np.float64).ravel()\n    it = np.asarray(s.get(\'it\', ()), np.float64).ravel()\n    n = min(len(mz), len(it))\n    mz, it = mz[:n], it[:n]\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    mz, it = mz[ok], it[ok]\n    if len(mz) == 0:\n        return None\n    p = c[\'intensity\']\n    iw = it if p == \'linear\' else np.sqrt(it) if p == \'sqrt\' else it ** float(p)\n    if c[\'mz_power\']:\n        prec = float(s.get(\'prec\') or mz.max())\n        iw = iw * (mz / prec) ** float(c[\'mz_power\'])\n    tot = iw.sum()\n    if not tot > 0:\n        return None\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\'adduct\'), s.get(\'mode\')) for dh in c[\'h_shifts\']])\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\n    return iw / tot, q, len(shifts), len(mz)\n\n\ndef _score_spectrum(levels, weights, prep, tol):\n    iw, q, ns, npk = prep\n    best = np.zeros(npk)\n    for arr, wt in zip(levels, weights):\n        if len(arr) == 0 or wt <= 0:\n            continue\n        a = np.searchsorted(arr, q - tol, \'left\')\n        b = np.searchsorted(arr, q + tol, \'right\')\n        hit = (b > a).reshape(ns, npk).any(axis=0)\n        best = np.where(hit & (wt > best), wt, best)\n    return float(iw @ best)\n\n\ndef candidate_levels(smi, cfg=None, cache=None):\n    """Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter)."""\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\n\n\ndef _levels_cached(smi, c, cache):\n    key = (smi, int(c[\'max_breaks\']), int(c[\'max_bonds_2\']), int(c[\'max_bonds_3\']))\n    if cache is not None and key in cache:\n        return cache[key]\n    try:\n        lv = fragment_levels(smi, c[\'max_breaks\'], c[\'max_bonds_2\'], c[\'max_bonds_3\'])\n    except Exception:\n        lv = None\n    if cache is not None:\n        cache[key] = lv\n    return lv\n\n\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\n    """One score per candidate: mean explained-intensity share over the molecule\'s spectra (None: unparsable)."""\n    c = resolve_cfg(cfg)\n    cache = {} if cache is None else cache\n    weights = [1.0, float(c[\'w1\']), float(c[\'w2\']), float(c[\'w3\'])][: int(c[\'max_breaks\']) + 1]\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\n    tol = float(c[\'tol\'])\n    out = []\n    for smi in cand_smiles:\n        lv = _levels_cached(smi, c, cache)\n        if lv is None:\n            out.append(None)\n            continue\n        if not preps:\n            out.append(0.0)\n            continue\n        out.append(float(np.mean([_score_spectrum(lv, weights, p, tol) for p in preps])))\n    return out\n\n\ndef warmup():\n    """Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it."""\n    fragment_levels(\'CCOC(=O)c1ccccc1O\')\n    return HAVE_NUMBA\n'
_post = "from __future__ import annotations\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nfrom functools import lru_cache\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\n\n@lru_cache(maxsize=30000)\ndef mol_formula(s):\n    try:\n        m = Chem.MolFromSmiles(str(s))\n        return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n    except Exception:\n        return None\n\ndef finite(x,default=np.nan):\n    try:\n        t=float(x)\n        return t if np.isfinite(t) else default\n    except (ValueError,TypeError):\n        return default\n\ndef select_spectra(frame,max_n=12):\n    out=[]\n    by_bucket={}\n    for r in frame.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs,dtype=float).ravel()\n        it=np.asarray(r.ms2_normalized_intensities,dtype=float).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        good=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if good.sum()<5:continue\n        order=np.argsort(it[good])[-120:]\n        mz,it=mz[good][order],it[good][order]\n        e=finite(getattr(r,'collision_energy_ev',30),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        item={'mz':mz,'it':it,'adduct':adduct,'prec':finite(getattr(r,'precursor_mz',None),float(mz.max())),'mode':mode}\n        bucket=(adduct,round(e/10))\n        by_bucket.setdefault(bucket,[]).append((int(good.sum()),item))\n    for key in sorted(by_bucket):\n        out += [v for _,v in sorted(by_bucket[key],key=lambda x:-x[0])[:2]]\n    return out[:max_n]\n\ndef support(payload,mid,s):\n    eng=(payload.get('engine2',{}).get(mid) or {}).get('smiles') or []\n    pc=(payload.get('pubchem',{}).get(mid) or {}).get('pc') or []\n    main=payload.get('base_tail',{}).get(mid) or []\n    fusion=payload.get('fusion_overflow',{}).get(mid) or []\n    def rank(v):return v.index(s)+1 if s in v else 0\n    return dict(engine2=rank(eng),pubchem=rank(pc),main=rank(main),fusion=rank(fusion))\n\ndef score_candidates(frag,spec,candidates,cache):\n    if not spec:return None\n    opt=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cache=cache)],float)\n    old=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cfg=frag.BASELINE_CFG,cache=cache)],float)\n    if not np.isfinite(opt).all() or not np.isfinite(old).all():return None\n    chunks=[spec[i:i+2] for i in range(0,len(spec),2)]\n    group=np.array([[np.nan if v is None else v for v in frag.score_molecule(ch,candidates,cache=cache)] for ch in chunks],float)\n    return opt,old,group\n\ndef decision(mid,control,listed,outs,query,payload,frag,cache,allow_tail=True,allow_top=True):\n    diag={'molecule_id':mid,'top1_changed':False,'tail_changed':False,'local_swaps':0,'reason':'unchanged','n_spectra':0,\n          'candidate_count':len(control),'top1_frag_margin':0.,'top1_neural_margin':0.,'new_candidate_source':''}\n    bysmiles=listed.set_index('smiles',drop=False)\n    if len(bysmiles)!=len(listed) or any(s not in bysmiles.index for s in control):\n        diag['reason']='evidence_mismatch';return control,diag\n    if not bool(listed.neural_ok.all()) or not bool(listed.processed.all()):\n        diag['reason']='unscored';return control,diag\n    n=len(control)\n    if n<3:return control,diag\n    anchor=mol_formula(control[0])\n    if anchor is None:\n        diag['reason']='formula_missing';return control,diag\n    cand=list(control[:min(12,n)])\n    outrows=[]\n    if allow_tail and outs is not None and len(outs) and n>=20:\n        for row in outs.sort_values('baseline_rank').itertuples(index=False):\n            if not bool(row.neural_ok) or not bool(row.processed):continue\n            s=str(row.smiles)\n            if s in control or mol_formula(s)!=anchor:continue\n            ev=support(payload,mid,s)\n            sources=int(bool(ev['engine2']))+int(bool(ev['pubchem']))+int(bool(ev['main']))\n            if sources<2 and not (0<ev['fusion']<=3):continue\n            outrows.append(row)\n            cand.append(s)\n            if len(outrows)>=8:break\n    spec=select_spectra(query)\n    diag['n_spectra']=len(spec)\n    if len(spec)<2:\n        diag['reason']='few_spectra';return control,diag\n    data=score_candidates(frag,spec,cand,cache)\n    if data is None:\n        diag['reason']='frag_unavailable';return control,diag\n    frag_new,frag_old,energy=data\n    pos={s:i for i,s in enumerate(cand)}\n    leader=control[0];lp=pos[leader]\n    neu_vals=np.asarray([finite(x) for x in listed.neural_score],float)\n    neural_sd=max(float(np.nanstd(neu_vals)),1e-6)\n    result=list(control)\n    lib=finite(bysmiles.loc[leader,'library_max'],0.)\n    if allow_top and lib<.87 and len(spec)>=3:\n        contenders=[]\n        for ix,s in enumerate(control[1:min(6,n)],1):\n            if mol_formula(s)!=anchor:continue\n            j=pos[s]; ev=bysmiles.loc[s]; lead_ev=bysmiles.loc[leader]\n            if not np.isfinite(frag_new[j]) or frag_new[j]<.27:continue\n            margin=float(frag_new[j]-frag_new[lp]); oldmargin=float(frag_old[j]-frag_old[lp])\n            if margin<.095 or oldmargin<.028:continue\n            votes=float((energy[:,j]>energy[:,lp]+.025).mean())\n            if votes<.70:continue\n            nmargin=(finite(ev.neural_score)-finite(lead_ev.neural_score))/neural_sd\n            if not np.isfinite(nmargin) or nmargin<.23:continue\n            if finite(ev.neural_cos)<finite(lead_ev.neural_cos)+.003 or finite(ev.neural_fp)<finite(lead_ev.neural_fp)-.012:continue\n            supports=0\n            for col in ('engine2_rank','pubchem_rank','main_rank'):\n                rank=finite(ev.get(col),0)\n                if 0<rank<=5:supports+=1\n            simulator_vote=0\n            for col in ('ice','gl'):\n                a,b=finite(ev.get(col)),finite(lead_ev.get(col))\n                if np.isfinite(a) and np.isfinite(b):\n                    if a>=b:simulator_vote+=1\n                    elif a<b-.35:simulator_vote-=1\n            if supports<1 and simulator_vote<1:continue\n            if simulator_vote<=-1:continue\n            contenders.append((margin+.05*nmargin+.025*simulator_vote+.01*supports,s,margin,nmargin))\n        if contenders:\n            _,winner,margin,nmargin=max(contenders,key=lambda v:v[0])\n            result.remove(winner);result.insert(0,winner)\n            diag.update(top1_changed=True,top1_frag_margin=round(margin,5),top1_neural_margin=round(nmargin,5),reason='fragment_neural_top1')\n    swaps=0\n    for i in range(2,min(11,n)):\n        prev,s=result[i-1],result[i]\n        if mol_formula(prev)!=mol_formula(s):continue\n        j,k=pos.get(s),pos.get(prev)\n        if j is None or k is None:continue\n        if frag_new[j]-frag_new[k]<.085 or frag_old[j]-frag_old[k]<.015:continue\n        if (energy[:,j]>energy[:,k]+.010).mean()<.65:continue\n        neui,neup=finite(bysmiles.loc[s,'neural_score']),finite(bysmiles.loc[prev,'neural_score'])\n        if not np.isfinite(neui) or neui<neup-.08*neural_sd:continue\n        result[i-1],result[i]=result[i],result[i-1]\n        swaps+=1\n        if swaps>=2:break\n    diag['local_swaps']=swaps\n    if not diag['top1_changed'] and swaps:diag['reason']='adjacent_fragment_consensus'\n    if allow_tail and outrows and len(result)==25:\n        last=result[-1]\n        if last in pos:\n            lastp=pos[last]\n            eligible=[]\n            listed_keys=set(listed.metric_key.dropna().astype(str))\n            for row in outrows:\n                s=str(row.smiles);j=pos[s]\n                if not isinstance(row.metric_key,str) or row.metric_key in listed_keys:continue\n                if not np.isfinite(frag_new[j]) or frag_new[j]<.28:continue\n                fm=float(frag_new[j]-frag_new[lastp])\n                if fm<.065 or frag_old[j]-frag_old[lastp]<.02:continue\n                if (energy[:,j]>energy[:,lastp]+.01).mean()<.65:continue\n                if finite(row.neural_score)<finite(bysmiles.loc[last,'neural_score'])+.22*neural_sd:continue\n                if finite(row.neural_cos)<=finite(bysmiles.loc[last,'neural_cos']) or finite(row.neural_fp)<=finite(bysmiles.loc[last,'neural_fp']):continue\n                ev=support(payload,mid,s)\n                supports=int(bool(ev['main']))+int(bool(ev['engine2']))+int(bool(ev['pubchem']))\n                eligible.append((fm,s,ev,supports))\n            if eligible:\n                margin,s,ev,supports=max(eligible,key=lambda q:q[0]+.015*q[3])\n                result[-1]=s\n                diag.update(tail_changed=True,new_candidate_source='+' .join(k for k in ('main','engine2','pubchem','fusion') if ev[k]),reason='tail_supported' if not diag['top1_changed'] and not swaps else diag['reason'])\n    assert len(result)==len(control) and len(set(result))==len(result)\n    assert set(result[:-1]).issubset(set(control)) and not (diag['top1_changed'] and result[0]==control[0])\n    return result,diag\n\ndef apply(root,comp,max_seconds=1500,allow_tail=True,allow_top=True):\n    import frag_rescore as frag\n    root=Path(root);comp=Path(comp)\n    control=root/'variants/submission_V2.csv'\n    if not control.is_file():raise FileNotFoundError(control)\n    src=pd.read_csv(control,dtype=str,keep_default_na=False)\n    need=['molecule_id','smiles','metric_key','processed','neural_ok','neural_score','neural_cos','neural_fp','baseline_rank']\n    listed=pd.read_parquet(root/'neural_candidate_features.parquet')\n    if not set(need).issubset(listed.columns):raise ValueError('Neural evidence columns missing')\n    outsiders_path=root/'outsider_neural_features.parquet'\n    outsider=pd.read_parquet(outsiders_path) if outsiders_path.is_file() else pd.DataFrame()\n    q=pd.read_parquet(root/'effective_query_spectra.parquet')\n    payload=json.loads((root/'e004_candidate_inputs.json').read_text())\n    lmap={str(mid):g.sort_values('baseline_rank') for mid,g in listed.groupby('molecule_id',sort=False)}\n    omap={str(mid):g for mid,g in outsider.groupby('molecule_id',sort=False)} if len(outsider) else {}\n    qmap={str(mid):g for mid,g in q.groupby('molecule_id',sort=False)}\n    start=time.monotonic()\n    cache={}; rows=[];audits=[];expired=False\n    frag.warmup()\n    for i,(mid,smiles) in enumerate(src.itertuples(index=False)):\n        mid=str(mid);old=smiles.split(';')\n        if time.monotonic()-start>max_seconds:\n            expired=True\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='time_budget',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        if mid not in lmap or mid not in qmap:\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='not_processed',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        try:\n            new,meta=decision(mid,old,lmap[mid],omap.get(mid),qmap[mid],payload,frag,cache,allow_tail=allow_tail,allow_top=allow_top)\n        except Exception as exc:\n            new,meta=old,dict(molecule_id=mid,reason='error:'+type(exc).__name__,top1_changed=False,tail_changed=False,local_swaps=0)\n        rows.append((mid,';'.join(new)));audits.append(meta)\n        if (i+1)%40==0:print('PARALLAX',i+1,'/',len(src),'elapsed',round(time.monotonic()-start,1),'s',flush=True)\n        if len(cache)>35000:cache.clear()\n    dest=pd.DataFrame(rows,columns=['molecule_id','smiles'])\n    ids=pd.read_parquet(comp/'test.parquet',columns=['molecule_id']).molecule_id.astype(str).unique().tolist()\n    if dest.molecule_id.tolist()!=src.molecule_id.tolist() or set(ids)!=set(dest.molecule_id):raise AssertionError('Molecule coverage changed')\n    for initial,new in zip(src.smiles,dest.smiles):\n        before=initial.split(';');after=new.split(';')\n        if not 1<=len(after)<=25 or len(after)!=len(set(after)) or any(not k for k in after):raise AssertionError('Submission format')\n        if after[0] not in before or not set(after[:-1]).issubset(set(before)):raise AssertionError('Unsafe candidate replacement')\n        if len(set(after)-set(before))>1:raise AssertionError('Too many outsiders')\n    out=root/'submission.csv';temp=root/'submission.parallax.tmp'\n    dest.to_csv(temp,index=False)\n    reread=pd.read_csv(temp,dtype=str,keep_default_na=False)\n    if not reread.equals(dest):raise AssertionError('CSV roundtrip failed')\n    temp.replace(out)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'spectral_parallax_audit.csv',index=False)\n    diagnostics={'control_sha256':hashlib.sha256(control.read_bytes()).hexdigest(),\n        'submission_sha256':hashlib.sha256(out.read_bytes()).hexdigest(),\n        'rows':len(dest),'changed_rows':int((src.smiles!=dest.smiles).sum()),\n        'top1_changed':int(sum(bool(r.get('top1_changed')) for r in audits)),\n        'tail_changed':int(sum(bool(r.get('tail_changed')) for r in audits)),\n        'adjacent_swaps':int(sum(int(r.get('local_swaps',0)) for r in audits)),\n        'time_budget_exceeded':expired,'elapsed_seconds':round(time.monotonic()-start,2),\n        'outcome':'experimental_unvalidated','baseline':'V2_0.468_user_reported'}\n    (root/'spectral_parallax_report.json').write_text(json.dumps(diagnostics,indent=2))\n    print('PARALLAX REPORT',json.dumps(diagnostics),flush=True)\n    return diagnostics\n"
assert __import__('hashlib').sha256(_physical.encode()).hexdigest() == 'eeacdcf0afb173a8b438054d93c30f061df7c9c69d1ae32db3530b2916f2e6a7'
assert __import__('hashlib').sha256(_post.encode()).hexdigest() == 'cd1594b3bbfa75ce30997559c5403438b7cec011879dcb9c0d48e111715dd85d'
(ROOT/'frag_rescore.py').write_text(_physical)
(CODE/'parallax_physical.py').write_text(_post)
print('Fragment source:', __import__('hashlib').sha256(_physical.encode()).hexdigest()[:12])


In [ ]:
BUNDLE = materialise_sources()
PIPELINE_DONE = run_public_pipeline()
if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
    shutil.copyfile(COMP / 'sample_submission.csv', ROOT / 'submission.csv')
    raise SystemExit('fast commit: sample file written, the scoring rerun runs every stage')
if not PIPELINE_DONE:
    raise RuntimeError('A real baseline run is required; no placeholder-only submission')
NEURAL_OK = score_final_lists()
if not NEURAL_OK:
    raise RuntimeError('The 0.468 V2 neural checkpoint is required; refusing silent 0.465 fallback')
OUTSIDERS_OK = score_outside_candidates(needed=True)
CONSENSUS_OK, REPORT, MOVES = build_consensus()
if not CONSENSUS_OK:
    raise RuntimeError('Unable to reproduce the V2 control')
SUBMITTED = choose_and_submit()
if SUBMITTED != VARIANT:
    raise RuntimeError(f'Expected {VARIANT}, got {SUBMITTED}')
shutil.copy2(ROOT/'submission.csv', ROOT/'submission_control_V2.csv')
print('Exact V2 control:', hashlib.sha256((ROOT/'submission_control_V2.csv').read_bytes()).hexdigest()[:16])


In [ ]:
import sys, traceback
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(CODE))
from parallax_physical import apply
try:
    _remaining = max(0, int(8.55 * 3600 - (time.time() - NB_T0)))
    if not PHYSICAL:
        PARALLAX_DIAGNOSTICS = {'status': 'disabled in this experiment'}
    elif _remaining < 180:
        PARALLAX_DIAGNOSTICS = {'status':'skipped_due_to_runtime_budget','remaining_seconds':_remaining,'control_retained':True}
    else:
        PARALLAX_DIAGNOSTICS = apply(ROOT, COMP, max_seconds=min(PHYSICAL_POST_SECONDS, _remaining - 120), allow_tail=True, allow_top=True)
except Exception:
    traceback.print_exc()
    shutil.copy2(ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
    PARALLAX_DIAGNOSTICS = {'error':'physical_overlay_failed', 'control_restored':True}
    (ROOT/'spectral_parallax_report.json').write_text(json.dumps(PARALLAX_DIAGNOSTICS,indent=2))
assert (ROOT/'submission.csv').is_file()
final = pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.notna().all() and final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(x.strip() for x in a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','rows',len(final),'control',SUBMITTED)
print('EXPERIMENT:', json.dumps(PARALLAX_DIAGNOSTICS))


In [ ]:
from pathlib import Path
import hashlib
_PUBLIC_RERANKER = "from __future__ import annotations\nimport json\nimport math\nimport time\nimport hashlib\nimport shutil\nfrom pathlib import Path\nfrom functools import lru_cache\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\nimport frag_rescore as frag\n\n@lru_cache(maxsize=40000)\ndef formula(s):\n    m = Chem.MolFromSmiles(str(s))\n    return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n\ndef number(x, default=float('nan')):\n    try:\n        v=float(x)\n        return v if math.isfinite(v) else default\n    except Exception:\n        return default\n\ndef member_keys(comp):\n    frame=pd.read_parquet(Path(comp)/'train.parquet',columns=['inchikey14'])\n    return set(frame.inchikey14.dropna().astype(str))\n\ndef balanced_spectra(query, max_groups=12):\n    groups={}\n    for r in query.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs, dtype=np.float64).ravel()\n        it=np.asarray(r.ms2_normalized_intensities, dtype=np.float64).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        ok=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if ok.sum()<6:continue\n        order=np.argsort(it[ok])[-120:]\n        mz,it=mz[ok][order],it[ok][order]\n        energy=number(getattr(r,'collision_energy_ev',30.),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        spec={'mz':mz,'it':it,'prec':number(getattr(r,'precursor_mz',None),float(mz.max())),\n              'adduct':adduct,'mode':mode}\n        bucket=(adduct,int(round(energy/10.)))\n        groups.setdefault(bucket,[]).append((int(ok.sum()), spec))\n    chosen=[]\n    for group in sorted(groups):\n        chosen.extend(s for _,s in sorted(groups[group],key=lambda x:-x[0])[:2])\n    return chosen[:max_groups]\n\ndef evidence_rows(df, order):\n    data=df.drop_duplicates('smiles').set_index('smiles',drop=False)\n    if any(s not in data.index for s in order):\n        return None\n    if ('neural_ok' in data and not data.loc[order,'neural_ok'].astype(bool).all()):\n        return None\n    if ('processed' in data and not data.loc[order,'processed'].astype(bool).all()):\n        return None\n    return data\n\ndef source_votes(candidate, leader):\n    good=0;bad=0\n    for label in ('engine2_rank','pubchem_rank','main_rank'):\n        c=number(candidate.get(label),0.);l=number(leader.get(label),0.)\n        if 0<c<=6:good+=1\n        if 0<l<=4 and (c<=0 or c>12):bad+=1\n    sim=0\n    for label in ('ice','gl'):\n        c=number(candidate.get(label));l=number(leader.get(label))\n        if math.isfinite(c) and math.isfinite(l):\n            if c>=l-.025:sim+=1\n            elif c<l-.30:sim-=1\n    return good,bad,sim\n\ndef paired_scores(spectra, smiles, cache):\n    matrix=[];matrix_old=[]\n    for spectrum in spectra:\n        vals=frag.score_molecule([spectrum],smiles,cache=cache)\n        old=frag.score_molecule([spectrum],smiles,cfg=frag.BASELINE_CFG,cache=cache)\n        if any(x is None or not math.isfinite(float(x)) for x in vals+old):return None\n        matrix.append(vals)\n        matrix_old.append(old)\n    return np.asarray(matrix,dtype=float),np.asarray(matrix_old,dtype=float)\n\ndef propose(order, original, data, query, train, cache):\n    record={'action':'none','top1_changed':False,'swaps':0,'n_spectra':0,\n            'protected_train':False,'physical_gain':None,'neural_gain_sd':None}\n    if len(order)<4 or not order or original[0]!=order[0]:\n        record['action']='protected_previous_top1';return order,record\n    evid=evidence_rows(data,order)\n    if evid is None:\n        record['action']='incomplete_evidence';return order,record\n    keys=[str(evid.loc[s,'metric_key']) if 'metric_key' in evid.columns else '' for s in order]\n    trained=np.asarray([k in train for k in keys],dtype=bool)\n    record['protected_train']=bool(trained.any())\n    leader=order[0]\n    if formula(leader) is None:\n        return order,record\n    lib=number(evid.loc[leader].get('library_max'),0.)\n    if lib>=.90:\n        record['action']='library_protected';return order,record\n    selected=balanced_spectra(query)\n    record['n_spectra']=len(selected)\n    if len(selected)<3:\n        record['action']='insufficient_spectra';return order,record\n    head=order[:min(10,len(order))]\n    p=paired_scores(selected,head,cache)\n    if p is None:\n        record['action']='fragment_unavailable';return order,record\n    a,b=p\n    neu=np.asarray([number(evid.loc[s].get('neural_score')) for s in order],float)\n    if not np.isfinite(neu).all():\n        record['action']='neural_unavailable';return order,record\n    ns=max(float(neu.std()),.00001)\n    updated=list(order)\n    candidates=[]\n    if not trained[0]:\n        for i in range(1,min(6,len(head))):\n            if trained[i] or formula(head[i])!=formula(leader):continue\n            new_margin=a[:,i]-a[:,0]\n            old_margin=b[:,i]-b[:,0]\n            mean=float(np.mean(new_margin));med=float(np.median(new_margin))\n            low=float(np.quantile(new_margin,.25));votes=float((new_margin>.020).mean())\n            old=float(np.mean(old_margin))\n            nz=float((neu[i]-neu[0])/ns)\n            if mean<.095 or med<.070 or low<-.010 or votes<.75 or old<.022 or nz<.18:continue\n            ev=evid.loc[head[i]];lv=evid.loc[leader]\n            if number(ev.get('neural_cos'),0.)<number(lv.get('neural_cos'),0.)+.002:continue\n            if number(ev.get('neural_fp'),0.)<number(lv.get('neural_fp'),0.)-.010:continue\n            votes_source,bad,sim=source_votes(ev,lv)\n            if votes_source<1 and sim<1:continue\n            if bad>=2 or sim<0:continue\n            confidence=mean+.045*nz+.020*sim+.010*votes_source+.018*max(0,low)\n            candidates.append((confidence,i,mean,nz))\n        if candidates:\n            _,idx,gain,nz=max(candidates)\n            updated.insert(0,updated.pop(idx))\n            record.update(action='provenance_spectral_top1',top1_changed=True,\n                          physical_gain=round(gain,5),neural_gain_sd=round(nz,4))\n    swaps=0\n    for pos in range(1,min(8,len(updated)-1)):\n        before=updated[pos];after=updated[pos+1]\n        if formula(before)!=formula(after):continue\n        i,j=head.index(before) if before in head else -1, head.index(after) if after in head else -1\n        if i<0 or j<0:continue\n        if trained[i] or trained[j]:continue\n        d=a[:,j]-a[:,i];do=b[:,j]-b[:,i]\n        if float(np.mean(d))<.068 or float(np.median(d))<.045 or float((d>.010).mean())<.67:continue\n        if float(np.mean(do))<.015:continue\n        if neu[j]-neu[i]<-.04*ns:continue\n        c=source_votes(evid.loc[after],evid.loc[before])[0]\n        if c<1 and neu[j]<=neu[i]:continue\n        updated[pos],updated[pos+1]=updated[pos+1],updated[pos]\n        swaps+=1\n        if swaps>=2:break\n    record['swaps']=swaps\n    if swaps and not record['top1_changed']:record['action']='paired_energy_swap'\n    # The private 0.475 model freezes known training candidates. This public analogue\n    # only allows permutations among untrained structures, preserving every training slot.\n    for i,s in enumerate(order):\n        if trained[i] and updated[i]!=s:\n            return order,dict(record,action='training_slot_guard',top1_changed=False,swaps=0)\n    return updated,record\n\ndef run(root,comp,time_budget=900):\n    root=Path(root);comp=Path(comp)\n    path=root/'submission.csv'\n    control=root/'submission_control_V2.csv'\n    for req in [path,control,root/'neural_candidate_features.parquet',root/'effective_query_spectra.parquet',comp/'train.parquet']:\n        if not req.exists():raise FileNotFoundError(req)\n    existing=pd.read_csv(path,dtype=str,keep_default_na=False)\n    first=pd.read_csv(control,dtype=str,keep_default_na=False)\n    assert existing.molecule_id.tolist()==first.molecule_id.tolist()\n    assert existing.molecule_id.is_unique\n    shutil.copy2(path,root/'submission_physical0472_control.csv')\n    nfeatures=pd.read_parquet(root/'neural_candidate_features.parquet')\n    spectra=pd.read_parquet(root/'effective_query_spectra.parquet')\n    groups={str(m):df for m,df in nfeatures.groupby('molecule_id',sort=False)}\n    specgroups={str(m):df for m,df in spectra.groupby('molecule_id',sort=False)}\n    known=member_keys(comp)\n    frag.warmup()\n    t0=time.monotonic();cache={};newrows=[];audits=[]\n    for idx,(record,ctrl) in enumerate(zip(existing.itertuples(index=False),first.itertuples(index=False))):\n        mid=str(record.molecule_id)\n        arr=str(record.smiles).split(';');original=str(ctrl.smiles).split(';')\n        if time.monotonic()-t0>time_budget:\n            new=arr;meta={'action':'runtime_budget','top1_changed':False,'swaps':0}\n        elif mid not in groups or mid not in specgroups:\n            new=arr;meta={'action':'no_source_spectra','top1_changed':False,'swaps':0}\n        else:\n            try:\n                new,meta=propose(arr,original,groups[mid],specgroups[mid],known,cache)\n            except Exception as e:\n                new=arr;meta={'action':'failed_'+type(e).__name__,'top1_changed':False,'swaps':0}\n        if set(new)!=set(arr) or len(new)!=len(arr):\n            raise AssertionError('Candidate identities may not change')\n        newrows.append((mid,';'.join(new)))\n        audits.append({'molecule_id':mid,**meta})\n        if len(cache)>25000:cache.clear()\n        if (idx+1)%50==0:\n            print('PROVENANCE',idx+1,'/',len(existing),'elapsed',round(time.monotonic()-t0,1),'s',flush=True)\n    result=pd.DataFrame(newrows,columns=['molecule_id','smiles'])\n    assert result.molecule_id.tolist()==existing.molecule_id.tolist()\n    assert result.molecule_id.is_unique\n    assert result.smiles.str.split(';').map(len).between(1,25).all()\n    assert result.smiles.str.split(';').map(lambda a:len(set(a))==len(a) and all(a)).all()\n    temp=root/'submission.provenance.tmp.csv'\n    result.to_csv(temp,index=False)\n    assert pd.read_csv(temp,dtype=str,keep_default_na=False).equals(result)\n    temp.replace(path)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'provenance_tournament_audit.csv',index=False)\n    summary={'source':'0.472 physical consensus public pipeline',\n             'private_rr3_weights_used':False,'train_structure_keys':len(known),\n             'rows':len(result),'changed':int((existing.smiles!=result.smiles).sum()),\n             'top1_changed':int(report.top1_changed.fillna(False).astype(bool).sum()),\n             'adjacent_swaps':int(report.swaps.fillna(0).sum()),\n             'elapsed_seconds':round(time.monotonic()-t0,2),\n             'control_hash':hashlib.sha256((root/'submission_physical0472_control.csv').read_bytes()).hexdigest(),\n             'final_hash':hashlib.sha256(path.read_bytes()).hexdigest(),\n             'note':'Leaderboard improvement not validated'}\n    (root/'provenance_tournament_report.json').write_text(json.dumps(summary,indent=2))\n    print('PROVENANCE_REPORT',json.dumps(summary),flush=True)\n    return summary\n"
PROVENANCE_CODE = ROOT/'parallax_public_provenance.py'
PROVENANCE_CODE.write_text(_PUBLIC_RERANKER)
assert hashlib.sha256(PROVENANCE_CODE.read_bytes()).hexdigest() == hashlib.sha256(_PUBLIC_RERANKER.encode()).hexdigest()
print('Public provenance-aware reranker:',PROVENANCE_CODE)

In [ ]:
import traceback
try:
    import importlib.util
    spec=importlib.util.spec_from_file_location('parallax_public_provenance',PROVENANCE_CODE)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    _left=max(0,int(8.65*3600-(time.time()-NB_T0)))
    if not PROVENANCE:
        PROVENANCE_REPORT={'status':'disabled in this experiment'}
    elif _left<240:
        PROVENANCE_REPORT={'status':'skipped_because_of_kaggle_runtime','seconds_remaining':_left}
    else:
        PROVENANCE_REPORT=module.run(ROOT,COMP,time_budget=min(960,_left-180))
except Exception as exc:
    traceback.print_exc()
    shutil.copy2(ROOT/'submission_physical0472_control.csv' if (ROOT/'submission_physical0472_control.csv').is_file() else ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
    PROVENANCE_REPORT={'status':'restored_control_after_error','exception':repr(exc)}

final=pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert list(final.columns)==['molecule_id','smiles']
assert final.molecule_id.tolist()==SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a))==len(a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','molecules',len(final),'provenance',PROVENANCE_REPORT)


In [ ]:
# ── Variant M: rank 1 from the fragment check on the V2 order, ranks 2-25 from CUSTOM (busyaprime experiment) ────────
# Fragment check: Roman Tamrazov's Spectral Parallax, https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax
# (cell 5 here is his, unpatched: it reads variants/submission_V2.csv, the input of his V17, the version with 0.472).
# CUSTOM 0.25/0.75: alex chilton, https://www.kaggle.com/code/alexchilton/casmi26-lb-0-472-leader-safe-ranks-2-25-at-0-75
# Rule, per molecule: P = submission.csv as the fragment check wrote it (V2 + physical), V = the V2 file, C = the CUSTOM
# file. A list is merged only when V[0] == C[0] and P holds the same candidates as C. If the check moved the leader
# (P[0] != V[0]), the answer is [P[0], V[0]] + the rest of C in C's order; otherwise it is C exactly. The check's
# adjacent swaps and lists it changed in any other way are dropped. On any error the CUSTOM file is restored.
# Merge rule from the judge panel of workflow wf_9745b485 (busyaprime, 2026-10-28).
import os, json, shutil, hashlib, traceback
import pandas as pd


def pxm_merge(physical, v2, custom):
    """physical, v2, custom: molecule_id, smiles frames in the same row order. Returns (frame, counts)."""
    assert list(physical.columns) == list(v2.columns) == list(custom.columns) == ['molecule_id', 'smiles']
    assert physical.molecule_id.tolist() == v2.molecule_id.tolist() == custom.molecule_id.tolist(), 'row order differs'
    rows = []
    counts = dict(rows=0, merged=0, leader_moves=0, skipped={}, skipped_leader_moves=0, moved_ids=[])
    for mid, ps, vs, cs in zip(custom.molecule_id, physical.smiles, v2.smiles, custom.smiles):
        p, v, c = ps.split(';'), vs.split(';'), cs.split(';')
        counts['rows'] += 1
        text = cs
        if v[0] != c[0]:
            reason = 'v2_custom_leader_differs'
        elif sorted(p) != sorted(c):
            reason = 'candidate_set_differs'
        else:
            reason = None
        if reason:
            counts['skipped'][reason] = counts['skipped'].get(reason, 0) + 1
            counts['skipped_leader_moves'] += int(p[0] != v[0])
        else:
            counts['merged'] += 1
            if p[0] != v[0]:
                new = [p[0], v[0]] + [s for s in c if s not in (p[0], v[0])]
                assert sorted(new) == sorted(c) and len(set(new)) == len(new), mid
                text = ';'.join(new)
                counts['leader_moves'] += 1
                counts['moved_ids'].append(mid)
        rows.append((mid, text))
    return pd.DataFrame(rows, columns=['molecule_id', 'smiles']), counts


# ---- run (everything above is definitions) ----
def _pxm_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()


def _pxm_main():
    rep = dict(status='disabled')
    if not globals().get('FRAG_ON_V2', False):
        return rep
    assert not globals().get('PROVENANCE', False), 'variant M is defined without the provenance reranker'
    sub_path = ROOT / 'submission.csv'
    v_path, c_path = ROOT / 'variants/submission_V2.csv', ROOT / 'variants/submission_CUSTOM.csv'
    rep['input_sha256'] = dict(physical_v2=_pxm_sha(sub_path), v2=_pxm_sha(v_path), custom=_pxm_sha(c_path))
    diag = globals().get('PARALLAX_DIAGNOSTICS') or {}
    rep['overlay'] = {k: diag[k] for k in ('status', 'error', 'control_sha256', 'submission_sha256', 'changed_rows',
                                           'top1_changed', 'tail_changed', 'adjacent_swaps') if k in diag}
    if diag.get('control_sha256'):
        assert diag['control_sha256'] == rep['input_sha256']['v2'], 'the fragment check did not read the V2 file'
    if diag.get('submission_sha256'):
        assert diag['submission_sha256'] == rep['input_sha256']['physical_v2'], \
            'submission.csv is not the file the fragment check wrote'
    frames = [pd.read_csv(p, dtype=str, keep_default_na=False) for p in (sub_path, v_path, c_path)]
    for f in frames:
        assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, \
            'an input file differs from sample_submission'
    new, counts = pxm_merge(*frames)
    assert new.molecule_id.tolist() == SAMPLE_IDS
    assert new.smiles.str.split(';').map(len).between(1, 25).all()
    assert new.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
    tmp = ROOT / 'submission.merge.tmp'
    if new.smiles.tolist() == frames[2].smiles.tolist():
        shutil.copyfile(c_path, tmp)              # final = C exactly: the CUSTOM bytes, not a re-serialisation
    else:
        new.to_csv(tmp, index=False)
    out_sha = _pxm_sha(tmp)
    os.replace(tmp, sub_path)
    rep.update(status='applied', **counts, output_sha256=out_sha,
               output_equals_custom=out_sha == rep['input_sha256']['custom'])
    return rep


try:
    MERGE_REPORT = _pxm_main()
except Exception as exc:
    traceback.print_exc()
    MERGE_REPORT = dict(status='restored_custom_after_error', exception=repr(exc)[:600])
    try:
        _tmp = ROOT / 'submission.merge.tmp'
        shutil.copyfile(ROOT / 'variants/submission_CUSTOM.csv', _tmp)
        os.replace(_tmp, ROOT / 'submission.csv')
        MERGE_REPORT['output_sha256'] = _pxm_sha(ROOT / 'submission.csv')
    except Exception as exc2:
        traceback.print_exc()
        MERGE_REPORT['restore_error'] = repr(exc2)[:300]
try:
    _tmp = ROOT / 'merge_report.json.tmp'
    _tmp.write_text(json.dumps(MERGE_REPORT, indent=2, default=str))
    os.replace(_tmp, ROOT / 'merge_report.json')
except Exception:
    traceback.print_exc()

final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
assert list(final.columns) == ['molecule_id', 'smiles']
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1, 25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
print('SUBMISSION READY:', ROOT / 'submission.csv', 'molecules', len(final), 'MERGE', json.dumps(
    {k: MERGE_REPORT.get(k) for k in ('status', 'rows', 'merged', 'leader_moves', 'skipped', 'skipped_leader_moves',
                                      'output_equals_custom', 'output_sha256', 'exception') if k in MERGE_REPORT},
    default=str))


## 4. What this run did

The table counts, layer by layer, how many answer lists changed against the layer before. In the commit run the
pipeline ranks only the first 12 visible molecules (the rest are placeholders), and the visible molecules are copies
of training spectra, so every leader has a library match of 1.0 and the fragment check's rank-1 rule cannot fire;
only its swaps can. The neural fusion can still move a leader (the V1 row). The last row is the merge: the 0.75 order with the fragment check's leader moves from the V2 order; with no leader move in the commit run it equals the 0.75 order. The hidden rerun that produced the
score runs every molecule and is not shown to anyone.

In [ ]:
# What each layer did in this run, read from the files the stages wrote.
if SHOW:
    rep = {}
    for name in ('spectral_parallax_report.json', 'provenance_tournament_report.json', 'merge_report.json'):
        p = ROOT / name
        if p.is_file():
            try:
                rep[name] = json.loads(p.read_text())
            except Exception as exc:  # noqa: BLE001 - a report is only shown, never used
                rep[name] = {'unreadable': repr(exc)}
    base = pd.read_csv(BASELINE, dtype=str, keep_default_na=False) if BASELINE.is_file() else None
    final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    layers = [('0.465 pipeline', BASELINE), ('V1, neural 0.25', VARIANTS_DIR / 'submission_V1.csv'),
              ('V2, ranks 2-25 at 0.50', VARIANTS_DIR / 'submission_V2.csv'),
              (f'{VARIANT}, ranks 2-25 at {CUSTOM["w_mid"]:.2f}', VARIANTS_DIR / f'submission_{VARIANT}.csv')]
    if globals().get('FRAG_ON_V2'):
        layers += [('submitted: CUSTOM + the fragment check\'s leader moves on the V2 order', ROOT / 'submission.csv')]
    elif PROVENANCE:
        layers += [('after the fragment check', ROOT / 'submission_physical0472_control.csv'),
                   ('after the provenance reranker (submitted)', ROOT / 'submission.csv')]
    else:
        layers += [('after the fragment check (submitted)' if PHYSICAL else 'submitted', ROOT / 'submission.csv')]
    prev, out = None, []
    for label, path in layers:
        if not path.is_file():
            continue
        f = pd.read_csv(path, dtype=str, keep_default_na=False).set_index('molecule_id').smiles
        if prev is not None:
            common = f.index.intersection(prev.index)
            moved = int((f[common] != prev[common]).sum())
            top1 = int(sum(a.split(';')[0] != b.split(';')[0] for a, b in zip(f[common], prev[common])))
            out.append({'layer': label, 'lists re-ordered vs previous': moved, 'rank 1 changed': top1})
        else:
            out.append({'layer': label, 'lists re-ordered vs previous': '-', 'rank 1 changed': '-'})
        prev = f
    table_html(pd.DataFrame(out).set_index('layer', drop=False), numeric=('lists re-ordered vs previous', 'rank 1 changed'))
    print('Commit run: the counts cover the 12 visible smoke molecules; the hidden scoring rerun is not shown to anyone.')
    for k, v in rep.items():
        print(k, json.dumps(v)[:600])

## 5. Every submission of this pipeline

Each row is a kernel on our account (busyaprime/33-casmi26-px-e1, e2, e3, e4, e5, e6, m1, m2, m4, m6, m5, m7, n2) running this pipeline on the hidden
test, submitted between 2026-10-27 and 2026-10-29 (our four earlier submissions, 2026-09-21, ran a different pipeline).
Besides the switches, e1-e6 carry one patch this version does not: their fragment check reads the 0.75 file
(variants/submission_CUSTOM.csv) instead of the V2 file, so it runs on the 0.75 order. This notebook is a new
version of m1, so the score on this page is m1's. e3-e5 point the neural bundle pin at Ahmed Berat Özer's long-trained `final_model.pt`; e6 adds his NX isomer reranker; m1 and every m, t and n run after it keep the fragment check on the V2 order, and each changes one setting of its leader rule (or adds NX) against m1.

| run | ranks 2-25 weight | fragment check | provenance reranker | neural model | other | public LB | submission |
|---|---:|---|---|---|---|---:|---|
| e1 | 0.75 | on, 0.75 order | off | Saner Turhaner v1 |  | 0.472 | 57713785 |
| e2 | 0.75 | on, 0.75 order | on | Saner Turhaner v1 |  | 0.472 | 57713786 |
| e3 | 0.75 | on, 0.75 order | off | long (Ahmed Berat Özer) |  | 0.453 | 57714284 |
| e4 | 0.75 | off | off | long (Ahmed Berat Özer) |  | 0.461 | 57715407 |
| e5 | 0.75 | on, 0.75 order | on | long (Ahmed Berat Özer) |  | 0.453 | 57715244 |
| e6 | 0.75 | on, 0.75 order | off | Saner Turhaner v1 | NX isomer reranker, weight 2 | 0.414 | 57720847 |
| m1 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only | **0.477** | 57722897 |
| m2 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; library gate 0.9 | 0.477 | 57731403 |
| m4 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; from 2 spectra | 0.477 | 57742127 |
| m6 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; old-config margin 0.015 | 0.477 | 57752007 |
| m5 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; vote step 0.015 | 0.473 | 57752070 |
| m7 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; new-config margin 0.08 | 0.473 | 57752139 |
| n2 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; NX isomer reranker on the V2 order | 0.469 | 57752191 |

How to read a difference: the leaderboard page says the public board uses about 31% of the test data, and
participants estimate the public split at about 125-130 molecules ([Roman Rozen](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures)), so one molecule
moved from rank 2 to rank 1 is worth about 0.004 and the board prints three decimals. This notebook's 0.477 is +0.005 against 0.472, the score Spectral Parallax shows and the score a fork running alex chilton's 0.75 weights shows (his own page shows 0.467): the size of about one net leader move from rank 2 to rank 1. The hidden lists are not shown, so how many leaders actually moved is not known.

A full scoring run takes hours. The only public timings are for the earlier E007 pipeline (0.440) that our base grew
from: about 2 h 38 min and, for a later version, 2 h 44 min, each a full run over the 400 visible molecules
([Kino Lutor](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/749251)).

## 6. Run it yourself

Join the competition, then Copy & Edit this notebook (the competition data and the datasets in the credits stay
attached). Keep the accelerator on GPU T4 and internet off. Run All in the editor is a smoke test: it ranks 12
visible molecules and gives no score. For a score, Save Version (our commit runs took 21-30 minutes on a T4), then
submit that version; the scoring rerun on the hidden test takes hours.

| switch | this notebook | what it does |
|---|---|---|
| `VARIANT`, `CUSTOM` | `CUSTOM`, w_top 0.25, w_mid 0.75 | the leader-safe weights; `V2` is the 0.25 / 0.50 recipe |
| `PHYSICAL` | `True` | Roman Tamrazov's fragment check |
| `PROVENANCE` | `False` | his provenance-aware reranker |
| `FRAG_ON_V2` | `True` | keep the fragment check's leader moves from the V2 order on top of the 0.75 order; False leaves the check's own V2 output |
| `COMMIT_MODE` | `'smoke'` | the commit run ranks 12 visible molecules; the scoring rerun ranks all |

Our string patches to his code, each asserted by our builder to apply exactly once:

1. Configuration cell: `VARIANT = 'CUSTOM'`, the CUSTOM weights, and the `PHYSICAL`, `PROVENANCE` and `FRAG_ON_V2`
   switches.
2. Fragment-check source (hidden): unchanged. It reads the V2 file, as upstream does.
3. Run cell (`BUNDLE = materialise_sources()`): it accepts the configured variant instead of only V2, and in a fast
   commit it writes the sample file and stops.
4. Fragment-check cell and provenance cell: an `if not PHYSICAL` and an `if not PROVENANCE` gate.
5. Our merge cell, after the provenance cell: per molecule, if the fragment check moved the V2 leader, the answer is
   [its new leader, the V2 leader] followed by the 0.75 order; otherwise the 0.75 order unchanged. It applies only
   when V2 and the 0.75 order share the leader and the fragment check's output holds the same candidates as the 0.75
   order; a list where the check also put an outside candidate at rank 25 keeps the 0.75 order without its leader
   move. Any error restores the 0.75 order.

## 7. What we concluded

1. The fragment check's leader moves from the V2 order on top of the 0.75 order (m1): 0.477, against 0.472 for the 0.75 order alone (its forks) and 0.472 for both changes stacked the plain way (e1 0.472).
2. The provenance reranker on top of the plain stack: e1 0.472 -> e2 0.472 (+0.000, no change at three decimals).
3. The fragment check run on the long-trained model's order: e4 0.461 -> e3 0.453 (-0.008, a loss at three decimals). This fits a check that depends on the order it is handed, but the pair switches all three parts of the check and does not single out the rank-1 rule.
4. Ahmed Berat Özer's long-trained neural model instead of Saner Turhaner's: e1 0.472 -> e3 0.453 (-0.019, a loss at three decimals). Its offline diagnostic (MRR 0.484 to 0.718 on sanpier's random holdout) did not carry to the hidden test.
5. Ahmed Berat Özer's NX isomer reranker at weight 2 on the plain stack: e1 0.472 -> e6 0.414 (-0.058, a loss at three decimals).
6. Opening the leader rule up to library match 0.90: m1 0.477 -> m2 0.477 (+0.000, no change at three decimals).
7. Opening the leader rule to molecules with two spectra: m1 0.477 -> m4 0.477 (+0.000, no change at three decimals).
8. The leader rule's old-config margin .028 -> .015: m1 0.477 -> m6 0.477 (+0.000, no change at three decimals).
9. The leader rule's vote step .025 -> .015: m1 0.477 -> m5 0.473 (-0.004, a loss at three decimals).
10. The leader rule's new-config margin .095 -> .08: m1 0.477 -> m7 0.473 (-0.004, a loss at three decimals).
11. Ahmed Berat Özer's NX isomer reranker on the V2 order, its leader moves merged: m1 0.477 -> n2 0.469 (-0.008, a loss at three decimals).
12. Rank 1 is where this lineage gains and loses. The public runs that let the neural model move the leader more freely scored lower: rank-1 weight 0.50 shows 0.448 and 1.00 shows 0.359 on the 0.472 base, and a uniform weight of 0.35 at offset 3 shows 0.447 on two accounts.

## 8. Credits

The answer lists come from other people's code, run with the patches listed in section 6:

- **Roman Tamrazov**, [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax), version 22: the notebook this one runs, including the
  fragment check and the provenance reranker. His notebook thanks Kozykorpesh's
  [LB [0.468] balanced acquisition evidence](https://www.kaggle.com/code/kozykappa/lb-0-468-balanced-acquisition-evidence) first, for the reproducible V2 consensus pipeline, the
  neural spectrum-structure ranking, the candidate-evidence exports and the evidence-guided candidate recovery, and
  credits Heliosli's [training-structure protection](https://www.kaggle.com/code/heliosli/happy-to-make-the-dataset-public-if-it-helps) for the provenance idea.
- **Roman Rozen**, [From Peaks to Structures](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures): the leader-safe consensus (`consensus.py`) and the
  outside-candidate collection (`outsiders.py`).
- **alex chilton**, [Leader-Safe, Ranks 2-25 at 0.75](https://www.kaggle.com/code/alexchilton/casmi26-lb-0-472-leader-safe-ranks-2-25-at-0-75): the 0.75 weight on ranks 2-25.
- **Saner Turhaner**, [PubChem + Neural025](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466): the spectrum-candidate model, the reciprocal-rank
  fusion and the six helper modules.
- **Dmitrii Gluzdov**, [The Data Behind Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures): the data notebook that assembles
  and documents the shared 0.465 pipeline, and the datasets that carry it.
- **Huseyin Emre Aksoy** ([profile](https://www.kaggle.com/huseyinemreaksoy)): `frag_rescore.py`, the untrained
  MetFrag-lite fragment scorer that both the fragment check and the pipeline's fragment-gap score run (Apache-2.0,
  reached through Ahmed Berat Özer's v7c notebook).
- **Ahmed Berat Özer**, [the long-schedule spectrum model](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-spectrum-model-long): Saner Turhaner's model trained 40 x 2,000 steps instead of 16 x 500, same code; runs e3-e5 use it.

The embedded 0.465 pipeline keeps its own full credits and licence table (its section 6), unchanged. Its main
components: canqiang's E006 (the complete inference base); Ahmed Berat Özer's main engine, rankers, fingerprint
weights, candidate pool and the Kaggle packages of ICEBERG and GLACIER (the ms-pred models of the Coley group);
prvsiyan, megayak and seyitkaangunes (second engine, analogue propagation, engine fusion); flexonafft (the N1
challenger); hengck23's DreaMS resource; and the further authors and public spectral libraries listed there.

| dataset | author | used for | licence |
|---|---|---|---|
| [dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets) | Dmitrii Gluzdov (a collection of other authors' public files, each folder with its own notices) | models, candidate sets and engine code of the pipeline | CC BY-NC-SA 4.0; each folder keeps its source licence (`ATTRIBUTION.md`) |
| [dmitriigluzdov/casmi26-pubchem-popularity-prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) | Dmitrii Gluzdov | PubChem popularity prior | Other: PubChem courtesy of the U.S. NLM; LOTUS CC0; NPAtlas CC BY-NC 4.0 |
| [dmitriigluzdov/casmi26-natural-product-panel-benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) | Dmitrii Gluzdov | natural-product panel | Other: competition data terms |
| [dmitriigluzdov/casmi26-fold-safe-fpnet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) | Dmitrii Gluzdov | fold-safe FPNet checkpoints | Other: checkpoint under competition data terms; scripts Apache 2.0 |
| [dmitriigluzdov/casmi26-natural-product-knowledge-table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) | Dmitrii Gluzdov | natural-product knowledge table | Other: LOTUS CC BY 4.0, NPAtlas CC BY-NC 4.0, PubChem counts |
| [sanpier/casmi-spectrum-model-v1](https://www.kaggle.com/datasets/sanpier/casmi-spectrum-model-v1) | Saner Turhaner | the spectrum-candidate model | trained on competition data (CC BY-NC 4.0) |
| [sanpier/catalogue-experiment-saner](https://www.kaggle.com/datasets/sanpier/catalogue-experiment-saner) | Saner Turhaner | attached as in Spectral Parallax; this run does not read it | as on its page |

The competition data is CC BY-NC 4.0, and the models above are trained on it: use them non-commercially, with
attribution. We wrote the field record, the figure and the audit cells.